# 10 — An agentic document reader (E10)

**Question.** B1 reads ColQwen2's top 4 pages once. When those pages miss part of the evidence (39 of 101 development questions), B1 answers 4 of them correctly. Reading deeper down the same ranking (E9) completes the evidence for only 8 of the 39. Can a reader that **writes its own search queries and chooses which pages to read**, within a fixed budget, do better?

**Design in one paragraph.** The agent starts where B1 ends: its first message holds B1's exact top 4 pages. At each step it must call one of three tools:
- `search_pages(query)`: ColQwen2 over every page of the document; it returns page numbers, not images.
- `read_pages(pages)`: shows up to 4 page images.
- `finish(answer, cited_pages, evidence)`: B1's exact answer contract.

The limits are 6 model calls, 12 distinct pages and a pre-call cost check, with a forced `finish` on the last call. Same reader as B1 (`gpt-6-luna`, temperature 0, `reasoning_effort: none`, 768-px images). Only the loop and the tools differ.

**Data.** The 118 development questions of the 14 held-out-14 documents (E6–E9). The MMLongBench-Doc test pool is used up and is not touched.

**Inputs:**
- frozen E5, E8 and E9 packages;
- the archived 768-px renders (`data/cache/mmlongbench_doc/page_images/short768-long1536/`), read-only, never re-rendered;
- the E5 page embeddings (`data/cache/e5/`);
- the pinned ColQwen2 checkpoint (`data/cache/hf/colqwen2-v1.0-hf@ddc07d2317c8/`).

**Outputs:**
- `data/processed/mmlongbench_doc/e10/`;
- runs in `experiments/runs/e10/`;
- the frozen package `experiments/frozen/e10/`.

**Modes** (`VISUALDOC_E10_MODE`):
- `offline` (default): nothing is sent.
- `smoke`: 6 agent episodes, not scored.
- `full`: the 354 planned items.

The paid modes need the frozen specification `experiments/specs/e10_agentic_reader_v1.json` and the owner's approval (given 2026-11-27, cumulative ceiling $3.50).

Following `AGENTS.md`, this notebook imports nothing from `src/visualdoc`. Every operation is rebuilt visibly and checked against the frozen runs. The design and its reasoning are in `docs/e10_agentic_reader_proposal.md`.

## Method

The full reasoning for every choice is in `docs/e10_agentic_reader_proposal.md`. In short:

| Choice | Why |
|---|---|
| **Warm start** with B1's exact top 4 | The agent never sees less than B1, so a loss can only come from its behaviour. Easy questions stay cheap (finish at call 1). |
| **Three tools**: search returns numbers, read returns images, finish = B1's schema | Search is nearly free and reading is costly, so the agent can look before it pays. Same parser and scorer as B1. |
| **No zoom or calculator** | One factor per experiment. E7 bundled four changes and could not tell which one hurt. |
| **6 calls, 12 pages, 4 per read**, pre-call cost check, forced finish | An agent's cost depends on its own choices. Hard limits plus a forced answer keep it bounded and scoreable. |
| **Same reader and settings as B1** | A gain cannot come from a stronger model. |
| **Hand-written loop, no framework** | Every request is fingerprinted and logged, with no hidden prompts or retries (and `AGENTS.md`). |
| **B1s same-session control + two agent replicates** | The reader alias drifts. E8/E9 measured 4–7 flips between identical runs. |
| **Process metric**: evidence recall of the pages the agent *saw* | Checks the mechanism (better searching), not just the score. |

**Hypotheses:**
- **H1:** AG beats B1s on answerable accuracy.
- **H2:** the pages AG saw hold more annotated evidence than B1's top 4.
- **H3:** the gain sits on the 39 questions whose evidence is incomplete in the top 4.

**New condition.** The search tool encodes queries with ColQwen2 on the Datalab CPU. E5 encoded on Apple MPS. The parity cell below measures the difference. The page embeddings are E5's own.

In [1]:
# Setup: paths, mode, constants. Nothing here calls an API or loads a model.
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from math import ceil, floor
from types import SimpleNamespace
import base64
import hashlib
import json
import os
import subprocess
import tempfile
import time
import uuid

import numpy as np
import pandas as pd
import tiktoken
from IPython.display import display

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
FROZEN_DIR = PROJECT_ROOT / "experiments" / "frozen"
SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "e10_agentic_reader_v1.json"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "e10"
RUNS_DIR = PROJECT_ROOT / "experiments" / "runs" / "e10"
E10_FROZEN = FROZEN_DIR / "e10"
MODEL_DIR = PROJECT_ROOT / "data" / "cache" / "hf" / "colqwen2-v1.0-hf@ddc07d2317c8"
E5_CACHE = PROJECT_ROOT / "data" / "cache" / "e5" / "colqwen2-v1.0-hf@ddc07d2317c8" / "af02fac2cb9b"
QUERY_CACHE = PROJECT_ROOT / "data" / "cache" / "e10" / "queries_cpu"   # git-ignored; agent queries, encoded on the CPU
E8_RUN = FROZEN_DIR / "e8" / "run_20261125T140433934010Z_4edb90e7_full"

REQUIRED_INPUTS = {
    FROZEN_DIR / "e5" / "tables" / "rankings.parquet": "restore experiments/frozen/e5 from git",
    FROZEN_DIR / "e5" / "tables" / "per_question.csv": "restore experiments/frozen/e5 from git",
    FROZEN_DIR / "e5" / "tables" / "render_manifest_heldout.csv": "restore experiments/frozen/e5 from git",
    E8_RUN / "execution_log.jsonl": "restore experiments/frozen/e8 from git",
    FROZEN_DIR / "e9_stage0" / "tables" / "stage0_per_question.csv": "restore experiments/frozen/e9_stage0 from git",
    E5_CACHE / "config.json": "the E5 embedding cache: restore it from the archive (docs/ssp_cloud.md)",
    PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "questions.parquet": "run 00_document_processing",
}
missing = [f"{path.relative_to(PROJECT_ROOT)} ({fix})" for path, fix in REQUIRED_INPUTS.items() if not path.exists()]
assert not missing, "Notebook 10 inputs missing:\n  " + "\n  ".join(missing)

MODE = os.environ.get("VISUALDOC_E10_MODE", "offline")
assert MODE in {"offline", "smoke", "full"}, f"unknown VISUALDOC_E10_MODE {MODE!r}"
TOP_K = 4                    # B1 reads 4 pages; the agent starts with the same 4
MAX_CALLS = 6                # model calls per episode (the 6th is a forced finish)
MAX_PAGES = 12               # distinct page images an episode may see, the 4 warm-start pages included
MAX_READ = 4                 # pages per read_pages call
SEARCH_K = 8                 # results returned by search_pages
MAX_OUTPUT_TOKENS = 800      # B1's output allowance, per call
EPISODE_CAP_USD = 0.03       # per-episode worst-case ceiling, checked before each call
BUDGET_USD = 3.50            # owner-approved cumulative ceiling for E10 (smoke + full), 2026-11-27
NOT_ANSWERABLE = "Not answerable"
SEED = 20261127
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT.name}. Mode: {MODE}. Outputs: {OUT_DIR.relative_to(PROJECT_ROOT)}")

Project root: Multimodal-Document-Intelligence. Mode: offline. Outputs: data/processed/mmlongbench_doc/e10


In [2]:
# Integrity: the frozen packages this notebook reads equal their SHA256SUMS.
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def verify_sha256sums(folder):
    entries = [line.split(maxsplit=1) for line in (folder / "SHA256SUMS").read_text().splitlines() if line.strip()]
    return {name: (folder / name).is_file() and sha256_bytes((folder / name).read_bytes()) == digest for digest, name in entries}


for name in ("e5", "e8", "e9", "e9_stage0"):
    checks = verify_sha256sums(FROZEN_DIR / name)
    assert checks and all(checks.values()), f"experiments/frozen/{name} differs from its SHA256SUMS"
    print(f"experiments/frozen/{name}: {len(checks)} files equal their SHA256SUMS")

# The specification, once frozen, must equal its freeze commit (the paid gate below refuses otherwise).
SPEC = json.loads(SPEC_PATH.read_text()) if SPEC_PATH.is_file() else None
SPEC_SHA256 = sha256_bytes(SPEC_PATH.read_bytes()) if SPEC else None
if SPEC:
    committed = subprocess.run(["git", "log", "-n", "1", "--format=%H", "--", str(SPEC_PATH.relative_to(PROJECT_ROOT))],
                               cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
    SPEC_FREEZE_COMMIT = committed or None
    if committed:
        frozen_bytes = subprocess.run(["git", "show", f"{committed}:{SPEC_PATH.relative_to(PROJECT_ROOT)}"], cwd=PROJECT_ROOT,
                                      capture_output=True, check=True).stdout
        assert frozen_bytes == SPEC_PATH.read_bytes(), "the E10 spec differs from its last commit: commit it or revert it"
    print(f"E10 spec: sha256 {SPEC_SHA256[:12]}…, committed at {SPEC_FREEZE_COMMIT and SPEC_FREEZE_COMMIT[:7]}")
else:
    SPEC_FREEZE_COMMIT = None
    print("E10 spec not written yet: paid modes are refused.")

experiments/frozen/e5: 11 files equal their SHA256SUMS
experiments/frozen/e8: 10 files equal their SHA256SUMS
experiments/frozen/e9: 15 files equal their SHA256SUMS
experiments/frozen/e9_stage0: 5 files equal their SHA256SUMS
E10 spec: sha256 de1b9f4e9a88…, committed at 25cd488


In [3]:
# Population: the 118 held-out-14 questions (E6-E9), their frozen ColQwen2 rankings, annotated evidence and E9 Stage 0 targets.
rankings = pd.read_parquet(FROZEN_DIR / "e5" / "tables" / "rankings.parquet")
v = rankings[(rankings["arm"] == "V") & (rankings["population"] == "held-out")].set_index("qid")
v = v.assign(pages=v["pages"].map(lambda p: [int(x) for x in p]), scores=v["scores"].map(lambda s: np.asarray(s, dtype=float)))
assert len(v) == 118 and v["doc_id"].nunique() == 14

evidence = pd.read_csv(FROZEN_DIR / "e5" / "tables" / "per_question.csv")
evidence = evidence[evidence["population"] == "held-out"].set_index("qid")
evidence["evidence_pages"] = evidence["evidence_pages"].map(lambda s: [int(x) for x in json.loads(s)])
assert len(evidence) == 101

stage0 = pd.read_csv(FROZEN_DIR / "e9_stage0" / "tables" / "stage0_per_question.csv").set_index("qid")
e8 = pd.read_csv(FROZEN_DIR / "e8" / "tables" / "evaluation_per_question.csv")
b1_e8 = e8[e8["arm"] == "L-B1"].set_index("qid")

_all_questions = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "questions.parquet")
HELDOUT14 = sorted(v["doc_id"].unique())
dev = _all_questions[_all_questions["doc_id"].isin(HELDOUT14)].set_index("qid").copy()
del _all_questions  # the test pool is dropped here, before any row is looked at
dev["evidence_pages"] = dev["evidence_pages"].map(lambda pages: [int(p) for p in pages])
assert set(dev.index) == set(v.index) and len(dev) == 118

q = v[["doc_id", "pages", "scores"]].join(b1_e8[["answerable"]])
q["question"] = dev["question"]
q["labelled"] = q.index.isin(evidence.index)
q["evidence_pages"] = evidence["evidence_pages"].reindex(q.index)
q["n_evidence"] = stage0["n_evidence"].reindex(q.index)
q["incomplete_at_4"] = stage0["incomplete"].reindex(q.index).fillna(False).astype(bool)
q["complete_at_8"] = stage0["complete_at_8"].reindex(q.index)
QIDS = sorted(q.index)
P0 = q[q["labelled"]]
print(f"{len(q)} questions in {q['doc_id'].nunique()} documents ({int(q['answerable'].sum())} answerable); "
      f"{len(P0)} with annotated evidence, of which {int(P0['incomplete_at_4'].sum())} are incomplete in the top 4 and "
      f"{int((P0['incomplete_at_4'] & (P0['complete_at_8'] == False)).sum())} stay incomplete in the top 8.")

118 questions in 14 documents (102 answerable); 101 with annotated evidence, of which 39 are incomplete in the top 4 and 31 stay incomplete in the top 8.


In [4]:
# B1's request, rebuilt visibly from the frozen E8 log and the frozen 768-px render manifest (as notebook 08 does).
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


manifest = pd.read_csv(FROZEN_DIR / "e5" / "tables" / "render_manifest_heldout.csv").set_index(["doc_id", "page"])
DOC_PAGES = {doc: sorted(int(p) for p in group.index.get_level_values("page")) for doc, group in manifest.groupby(level="doc_id")}
e8_log = read_jsonl(E8_RUN / "execution_log.jsonl")
e8_logged = {(record["qid"], record["arm"]): record for record in e8_log}
assert len(e8_logged) == 354


def png_size(png):
    assert png.startswith(b"\x89PNG\r\n\x1a\n") and len(png) >= 24, "not a PNG"
    return int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")


def page_png(doc_id, page):
    """The frozen 768-px render of one page, refused if its bytes changed since E5."""
    row = manifest.loc[(doc_id, int(page))]
    png = (PROJECT_ROOT / row["image_path"]).read_bytes()
    assert sha256_bytes(png) == row["sha256"], f"stale render: {doc_id} p.{page}"
    return png


def image_part(png):
    return {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"), "detail": "high"}}


def page_parts(doc_id, pages):
    """'[Page N]' then the page image, for each page in the given order (B1's layout)."""
    parts = []
    for page in pages:
        parts.append({"type": "text", "text": f"[Page {page}]"})
        parts.append(image_part(page_png(doc_id, page)))
    return parts


def build_b1_request(qid):
    template = e8_logged[(qid, "L-B1")]["request"]
    header = template["messages"][1]["content"][0]
    assert header["type"] == "text" and header["text"].startswith("Question: ")
    content = [dict(header)] + page_parts(q.at[qid, "doc_id"], q.at[qid, "pages"][:TOP_K])
    return {**{key: value for key, value in template.items() if key != "messages"},
            "messages": [dict(template["messages"][0]), {"role": "user", "content": content}]}


def loggable(request):
    """The request with each image replaced by its SHA-256, byte length, dimensions and detail: safe to log, no Base64."""
    messages = []
    for message in request["messages"]:
        if not isinstance(message.get("content"), list):
            messages.append(dict(message))
            continue
        parts = []
        for part in message["content"]:
            if part["type"] == "image_url":
                png = base64.b64decode(part["image_url"]["url"].split(",", 1)[1])
                parts.append({"type": "image", "sha256": sha256_bytes(png), "bytes": len(png),
                              "dimensions": list(png_size(png)), "detail": part["image_url"].get("detail")})
            else:
                parts.append(part)
        messages.append({**{k: val for k, val in message.items() if k != "content"}, "content": parts})
    return {**{key: value for key, value in request.items() if key != "messages"}, "messages": messages}


def fingerprint(request):
    return sha256_bytes(json.dumps(loggable(request), sort_keys=True).encode())


B1_REQUESTS = {}
for qid in QIDS:
    request = build_b1_request(qid)
    assert loggable(request) == e8_logged[(qid, "L-B1")]["request"], f"{qid}: B1s differs from frozen E8 L-B1"
    assert fingerprint(request) == e8_logged[(qid, "L-B1")]["fingerprint"]
    B1_REQUESTS[qid] = {"fingerprint": fingerprint(request), "loggable": loggable(request)}
    del request
B1_SYSTEM_PROMPT = e8_logged[(QIDS[0], "L-B1")]["request"]["messages"][0]["content"]
B1_SCHEMA = e8_logged[(QIDS[0], "L-B1")]["request"]["response_format"]["json_schema"]["schema"]
print(f"Rebuilt {len(B1_REQUESTS)} B1s requests: each equals the frozen E8 L-B1 request byte for byte (loggable form and fingerprint).")

Rebuilt 118 B1s requests: each equals the frozen E8 L-B1 request byte for byte (loggable form and fingerprint).


In [5]:
# The measured gpt-6-luna token model (frozen E8 smoke), and a strict UPPER BOUND for multi-turn tool requests.
# The E8 model is exact for B1's single-turn request. Tool definitions and tool-call messages were never measured, so for agent
# calls the pre-call check uses a bound that cannot undercount: exact image tokens + the UTF-8 bytes of every text and JSON field
# (o200k_base is byte-level BPE: never more tokens than bytes) + 64 framing tokens per message + the tool-schema bytes.
MODEL = "gpt-6-luna"
PRICE = {"input": 0.10 / 1e6, "cached_input": 0.01 / 1e6, "output": 0.50 / 1e6}
CONTEXT_TOKENS = 128_000
ENCODING = tiktoken.get_encoding("o200k_base")


def luna_image_tokens(width, height):
    return floor(1.2 * ceil(width / 32) * ceil(height / 32))


def b1_prompt_tokens(logged_request):
    """Exact for B1's single-turn request (E8 smoke rule)."""
    texts, dims = [], []
    for message in logged_request["messages"]:
        parts = [{"type": "text", "text": message["content"]}] if isinstance(message["content"], str) else message["content"]
        for part in parts:
            if part["type"] == "text":
                texts.append(part["text"])
            elif part["type"] == "image":
                dims.append(tuple(part["dimensions"]))
    framing = 9 if dims else 10
    schema = 100 if logged_request.get("response_format", {}).get("type") == "json_schema" else 0
    return sum(len(ENCODING.encode(t)) for t in texts) + framing + schema + sum(luna_image_tokens(*d) for d in dims)


billed = [(r["prompt_tokens"], b1_prompt_tokens(r["request"])) for r in e8_log if r["arm"] in ("L-B1", "L-B1r")]
assert len(billed) == 236 and all(observed == estimated for observed, estimated in billed), "token model no longer reproduces E8"


def message_texts_and_images(logged_request):
    texts, dims = [], []
    for message in logged_request["messages"]:
        content = message.get("content")
        if isinstance(content, str):
            texts.append(content)
        elif isinstance(content, list):
            for part in content:
                (dims.append(tuple(part["dimensions"])) if part["type"] == "image" else texts.append(part["text"]))
        for call in message.get("tool_calls") or []:
            texts.append(call["function"]["name"])
            texts.append(call["function"]["arguments"])
    return texts, dims


def agent_prompt_bound(logged_request):
    texts, dims = message_texts_and_images(logged_request)
    tools = json.dumps(logged_request.get("tools", []))
    return (sum(len(t.encode("utf-8")) for t in texts) + len(tools.encode("utf-8")) + 64 * len(logged_request["messages"])
            + sum(luna_image_tokens(*d) for d in dims))


def agent_prompt_estimate(logged_request):
    """Best estimate (exact text tokens + images), reported beside the billed count; it is NOT used for the budget."""
    texts, dims = message_texts_and_images(logged_request)
    return sum(len(ENCODING.encode(t)) for t in texts) + sum(luna_image_tokens(*d) for d in dims)


def upper_cost(logged_request, prompt_bound=None):
    prompt = prompt_bound if prompt_bound is not None else (
        agent_prompt_bound(logged_request) if "tools" in logged_request else b1_prompt_tokens(logged_request))
    return prompt * PRICE["input"] + logged_request["max_completion_tokens"] * PRICE["output"]


for qid, spec in B1_REQUESTS.items():
    spec["prompt_tokens"] = b1_prompt_tokens(spec["loggable"])
    spec["upper_usd"] = upper_cost(spec["loggable"])
    assert agent_prompt_bound({**spec["loggable"], "tools": []}) >= spec["prompt_tokens"], "the bound must never undercount"
print("The E8 token model reproduces all 236 billed E8 L-B1/L-B1r prompt totals exactly; "
      "the agent bound is never below the exact count on the 118 B1 requests.")

The E8 token model reproduces all 236 billed E8 L-B1/L-B1r prompt totals exactly; the agent bound is never below the exact count on the 118 B1 requests.


In [6]:
# B1's reply parser, rebuilt as in notebook 08 and checked against all 236 frozen E8 B1 parses. The agent's finish arguments use it too.
def page_number(value):
    """A page >= 1: a JSON integer, a whole float (2.0), or ASCII digits ("4"). Never rounded."""
    if isinstance(value, int) and not isinstance(value, bool):
        number = value
    elif isinstance(value, float) and value.is_integer():
        number = int(value)
    elif isinstance(value, str) and value.isascii() and value.isdigit():
        number = int(value)
    else:
        raise ValueError(f"cited page {value!r} is not a page number")
    if number < 1:
        raise ValueError(f"cited page {value!r} is not a page number (pages start at 1)")
    return number


def valid_answer(answer):
    scalar = lambda x: isinstance(x, (str, int, float)) and not isinstance(x, bool)
    return scalar(answer) or (isinstance(answer, list) and all(scalar(item) for item in answer))


def parse_b1_reply(raw):
    """(answer, cited pages, evidence, error) under contract b1; any breach makes the whole reply unparseable."""
    try:
        payload = json.loads(raw)
        if not isinstance(payload, dict) or set(payload) != {"answer", "cited_pages", "evidence"}:
            raise TypeError("B1 output must contain exactly answer, cited_pages and evidence")
        answer = payload["answer"]
        if not valid_answer(answer):
            raise TypeError("answer is not a string, number, or list of strings/numbers")
        if not isinstance(payload["cited_pages"], list) or not isinstance(payload["evidence"], list):
            raise TypeError("cited_pages and evidence must be lists")
        cited = [page_number(page) for page in payload["cited_pages"]]
        evidence_items = []
        for item in payload["evidence"]:
            if not isinstance(item, dict) or set(item) != {"page", "quote"} or not isinstance(item["quote"], str):
                raise TypeError("each evidence item must contain exactly page and a string quote")
            evidence_items.append({"page": page_number(item["page"]), "quote": item["quote"]})
        if isinstance(answer, str) and answer.strip().lower() == "not answerable":
            answer = NOT_ANSWERABLE
        if answer == NOT_ANSWERABLE and (cited or evidence_items):
            raise ValueError("Not answerable must have empty cited_pages and evidence")
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        return "", [], [], f"unparseable model output: {error}"
    return answer, cited, evidence_items, None


for record in e8_log:
    if record["arm"] in ("L-B1", "L-B1r"):
        answer, cited, ev, error = parse_b1_reply(record["raw_response"])
        assert error == record["parse_error"] and record["parsed"] == {"answer": answer, "cited_pages": cited, "evidence": ev}
assert parse_b1_reply('{"answer": "x", "cited_pages": [2.7], "evidence": []}')[3] is not None
assert parse_b1_reply('{"answer": "not answerable", "cited_pages": [3], "evidence": []}')[3] is not None
print("parse_b1_reply reproduces all 236 frozen E8 B1 parses; it rejects fractional pages and cited abstentions.")

parse_b1_reply reproduces all 236 frozen E8 B1 parses; it rejects fractional pages and cited abstentions.


## Stage 0 — the search encoder on the CPU, and its parity with E5

In [7]:
# The search tool's encoder: ColQwen2 v1.0 (E5's checkpoint), query side only, on the Datalab CPU.
# E5 encoded on Apple MPS; the CPU is a NEW CONDITION for agent queries. The parity check below measures what it changes.
# Page embeddings are never recomputed: every page comes from E5's frozen cache (bf16, keyed by the render's SHA-256).
from safetensors.torch import load_file as load_safetensors, save_file as save_safetensors

E5_IDENTITY = {"repo": "vidore/colqwen2-v1.0-hf", "revision": "ddc07d2317c80f75fc742b7362ee9ad1912908f9",
               "weights_sha256": "2f6dbf32fc2db5b045d74ae73b148425d6e03cc1106a047cdcc7e72b6dc53aba",
               "preprocessor_config_sha256": "be69973e9e1f8404b6fb70f246ad8307086d614b31464e3048c387945ac66bf0",
               "tokenizer_sha256": "091aa7594dc2fcfbfa06b9e3c22a5f0562ac14f30375c13af7309407a0e67b8a",
               "torch": "2.15.0", "transformers": "5.23.0"}
e5_config = json.loads((E5_CACHE / "config.json").read_text())
assert all(e5_config[key] == value for key, value in E5_IDENTITY.items()), "E5's recorded encoder differs from this notebook's pin"

import torch  # noqa: E402  (CPU build in .venv; imported here, not at setup, so that offline replay of the tables never needs it)

PAGE_EMB = {doc: {int(page): load_safetensors(str(E5_CACHE / "pages" / f"{manifest.at[(doc, page), 'sha256']}.safetensors"))["embedding"].float()
                  for page in DOC_PAGES[doc]} for doc in HELDOUT14}
assert sum(len(pages) for pages in PAGE_EMB.values()) == 647


def maxsim(query, page):
    """Late interaction (notebook 04): each query vector's best page vector, summed."""
    return float((query @ page.T).max(dim=1).values.sum())


def rank_pages(doc_id, query_embedding):
    scored = [(page, maxsim(query_embedding, emb)) for page, emb in PAGE_EMB[doc_id].items()]
    return sorted(scored, key=lambda item: (-item[1], item[0]))


class QueryEncoder:
    """Loads the checkpoint on first use, after checking it is E5's (files, versions); caches every query embedding in bf16."""

    def __init__(self, model_dir=MODEL_DIR, cache_dir=QUERY_CACHE, threads=8):
        self.model_dir, self.cache_dir, self.threads = Path(model_dir), Path(cache_dir), threads
        self.model = self.processor = None
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _load(self):
        import transformers
        from transformers import ColQwen2ForRetrieval, ColQwen2Processor
        assert (torch.__version__.split("+")[0], transformers.__version__) == (E5_IDENTITY["torch"], E5_IDENTITY["transformers"]), \
            f"torch {torch.__version__} / transformers {transformers.__version__} differ from E5's: a new condition"
        for name, key in (("preprocessor_config.json", "preprocessor_config_sha256"), ("tokenizer.json", "tokenizer_sha256")):
            assert sha256_bytes((self.model_dir / name).read_bytes()) == E5_IDENTITY[key], f"{name} differs from E5's"
        weights = self.model_dir / "model.safetensors"
        record = self.cache_dir.parent / "weights_check.json"
        stat = weights.stat()
        checked = json.loads(record.read_text()) if record.is_file() else {}
        if checked.get("size") != stat.st_size or checked.get("mtime_ns") != stat.st_mtime_ns:
            digest = hashlib.sha256()
            with weights.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1 << 24), b""):
                    digest.update(chunk)
            checked = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns, "sha256": digest.hexdigest()}
            record.write_text(json.dumps(checked))
        assert checked["sha256"] == E5_IDENTITY["weights_sha256"], "the ColQwen2 weights differ from E5's"
        torch.set_num_threads(self.threads)
        self.processor = ColQwen2Processor.from_pretrained(self.model_dir)
        self.model = ColQwen2ForRetrieval.from_pretrained(self.model_dir, dtype=torch.bfloat16, attn_implementation="sdpa").eval()

    def __call__(self, text):
        """(tokens, 128) float32 of the bf16 embedding: 'Query: ' + text + 10 augmentation tokens (the processor's rule)."""
        path = self.cache_dir / f"{sha256_bytes(text.encode())}.safetensors"
        if path.is_file():
            return load_safetensors(str(path))["embedding"].float()
        if self.model is None:
            self._load()
        with torch.inference_mode():
            batch = self.processor(text=[text], return_tensors="pt")
            embedding = self.model(**batch).embeddings[0][batch["attention_mask"][0].bool()].contiguous()
        save_safetensors({"embedding": embedding}, str(path))
        return embedding.float()


encoder = QueryEncoder()  # 8 threads: the Datalab container has an 8-CPU quota (cgroup cpu.max); more threads only throttle
print(f"Page embeddings for {len(PAGE_EMB)} documents ({sum(len(p) for p in PAGE_EMB.values())} pages) loaded from E5's cache; "
      f"query cache {QUERY_CACHE.relative_to(PROJECT_ROOT)} holds {len(list(QUERY_CACHE.glob('*.safetensors')))} embeddings.")

Page embeddings for 14 documents (647 pages) loaded from E5's cache; query cache data/cache/e10/queries_cpu holds 369 embeddings.


In [8]:
# Parity of the CPU query encoder with E5 (MPS), on the 118 original questions. The pages are E5's own embeddings, so any
# difference comes from the query side. Gate, fixed before this cell first ran (2026-11-27):
#   the top-4 page SET equals frozen E5's for >= 90 % of questions, and evidence recall@4 differs by at most 0.02.
# The agent's warm start does not depend on this (it uses E5's frozen top 4); only search_pages does.
PARITY_PATH = OUT_DIR / "stage0_parity.csv"
PARITY_GATE = {"top4_set_share_min": 0.90, "recall_at_4_abs_diff_max": 0.02}


def evidence_recall(pages, evidence_pages):
    return len(set(pages) & set(evidence_pages)) / len(set(evidence_pages))


if PARITY_PATH.is_file() and os.environ.get("VISUALDOC_E10_PARITY") != "recompute":
    parity = pd.read_csv(PARITY_PATH).set_index("qid")
    print(f"Parity table read from {PARITY_PATH.relative_to(PROJECT_ROOT)} (set VISUALDOC_E10_PARITY=recompute to re-encode).")
else:
    rows, started = [], time.perf_counter()
    for qid in QIDS:
        ranked = [page for page, _ in rank_pages(q.at[qid, "doc_id"], encoder(q.at[qid, "question"]))]
        frozen = q.at[qid, "pages"]
        row = {"qid": qid, "full_equal": ranked == frozen, "top4_list": ranked[:4] == frozen[:4],
               "top4_set": set(ranked[:4]) == set(frozen[:4]), "top8_list": ranked[:8] == frozen[:8],
               "top4_overlap": len(set(ranked[:4]) & set(frozen[:4])), "cpu_top8": json.dumps(ranked[:8])}
        if q.at[qid, "labelled"]:
            row["recall4_cpu"] = evidence_recall(ranked[:4], q.at[qid, "evidence_pages"])
            row["recall4_e5"] = evidence_recall(frozen[:4], q.at[qid, "evidence_pages"])
        rows.append(row)
    parity = pd.DataFrame(rows).set_index("qid")
    parity.to_csv(PARITY_PATH)
    print(f"Encoded and ranked {len(parity)} questions on the CPU in {time.perf_counter() - started:.0f} s.")

parity_summary = {
    "full ranking equal": int(parity["full_equal"].sum()), "top 8 order equal": int(parity["top8_list"].sum()),
    "top 4 order equal": int(parity["top4_list"].sum()), "top 4 set equal": int(parity["top4_set"].sum()),
    "mean top-4 overlap": round(float(parity["top4_overlap"].mean()), 3),
    "recall@4 CPU": round(float(parity["recall4_cpu"].mean()), 4), "recall@4 E5": round(float(parity["recall4_e5"].mean()), 4)}
display(pd.Series(parity_summary, name=f"CPU vs E5 (n = {len(parity)})").to_frame())
PARITY_PASSED = (parity["top4_set"].mean() >= PARITY_GATE["top4_set_share_min"]
                 and abs(parity["recall4_cpu"].mean() - parity["recall4_e5"].mean()) <= PARITY_GATE["recall_at_4_abs_diff_max"])
assert PARITY_PASSED, "the CPU encoder fails the parity gate: E10 cannot use it as the search tool"
print("Parity gate passed: the CPU query encoder is a faithful ColQwen2 search tool for this experiment (a recorded new condition).")

Parity table read from data/processed/mmlongbench_doc/e10/stage0_parity.csv (set VISUALDOC_E10_PARITY=recompute to re-encode).


,CPU vs E5 (n = 118)
full ranking equal,25.0000
top 8 order equal,92.0000
top 4 order equal,112.0000
top 4 set equal,113.0000
mean top-4 overlap,3.9580
recall@4 CPU,0.7234
recall@4 E5,0.7259


Parity gate passed: the CPU query encoder is a faithful ColQwen2 search tool for this experiment (a recorded new condition).


## Stage 1–2 — the tools, the loop, and its tests without a model

In [9]:
# The agent's contract: instructions, three tools, and the first message (warm start = B1's exact top 4 pages).
AGENT_PROMPT_VERSION = "e10-agent-v1"
AGENT_SYSTEM_PROMPT = """You answer questions about a single PDF document using only its page images.
You work in steps. At every step you must call exactly one tool.

You start with the 4 pages that a visual search engine ranked highest for the question. They may not contain all the evidence.

Tools:
- search_pages(query): ranks every page of the document by visual similarity to your query and returns the best page numbers,
  with scores relative to the best result. It does not show the pages. Write a short, specific query with words that are likely
  printed on the evidence page: a table or chart title, a section heading, a label, a name, a year. When the question asks about
  several things (a comparison, a total, a list across sections), search for each part separately.
- read_pages(pages): shows you up to 4 page images, each preceded by [Page N].
- finish(answer, cited_pages, evidence): your final answer.

How to work:
- If the pages you have already seen contain everything needed, call finish at once.
- Otherwise search, read the most promising pages, and finish when every part of the answer is supported by a page you have seen.
- Each tool result reports your remaining budget. Your last step must be finish.

Rules for finish:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the pages you have seen do not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": the page numbers N of pages you have seen whose images support the answer ([] if "Not answerable").
  "evidence": for each cited page, one short passage copied verbatim from that page (at most 30 words)
              that supports the answer ([] if "Not answerable").

Do not use knowledge outside the page images. Do not guess."""

TOOLS = [
    {"type": "function", "function": {
        "name": "search_pages", "strict": True,
        "description": "Rank every page of the document by visual similarity to a short query. Returns page numbers and relative scores, not images.",
        "parameters": {"type": "object", "additionalProperties": False, "required": ["query"],
                       "properties": {"query": {"type": "string", "description": "A short, specific search query."}}}}},
    {"type": "function", "function": {
        "name": "read_pages", "strict": True,
        "description": f"Show up to {MAX_READ} page images, each preceded by [Page N].",
        "parameters": {"type": "object", "additionalProperties": False, "required": ["pages"],
                       "properties": {"pages": {"type": "array", "items": {"type": "integer"},
                                                "description": f"1-based page numbers, at most {MAX_READ}."}}}}},
    {"type": "function", "function": {
        "name": "finish", "strict": True,
        "description": "Give the final answer. This ends the episode.",
        "parameters": B1_SCHEMA}},
]
TOOL_NAMES = [tool["function"]["name"] for tool in TOOLS]
FORCE_FINISH = {"type": "function", "function": {"name": "finish"}}
REQUEST_PARAMS = {"model": MODEL, "max_completion_tokens": MAX_OUTPUT_TOKENS, "reasoning_effort": "none", "temperature": 0,
                  "parallel_tool_calls": False}
AGENT_CONTRACT_SHA256 = sha256_bytes(json.dumps({"version": AGENT_PROMPT_VERSION, "system": AGENT_SYSTEM_PROMPT, "tools": TOOLS,
                                                 "params": REQUEST_PARAMS}, sort_keys=True).encode())
# The answer rules are B1's, apart from "pages you have seen" in place of "page images" (the agent's pages vary), and the
# finish schema is B1's strict schema object itself, so both arms are parsed and scored identically.
assert TOOLS[2]["function"]["parameters"] is B1_SCHEMA


def budget_line(calls_used, seen):
    calls_left = MAX_CALLS - calls_used
    pages_left = MAX_PAGES - len(seen)
    if calls_left <= 1:
        return f"Budget: this was your last tool step. Your next step must be finish. Pages you may still read: {pages_left}."
    return f"Budget: {calls_left} steps left, the last of which must be finish. Pages you may still read: {pages_left}."


def first_message(qid):
    doc_id, question = q.at[qid, "doc_id"], q.at[qid, "question"]
    warm = q.at[qid, "pages"][:TOP_K]
    header = (f"Question: {question}\n\nThe document has {len(DOC_PAGES[doc_id])} pages. The 4 pages the visual search engine ranked "
              f"highest for the question are shown below, best first.\n\nPage images:")
    return {"role": "user", "content": [{"type": "text", "text": header}] + page_parts(doc_id, warm)
            + [{"type": "text", "text": budget_line(0, warm)}]}


def normalise_query(text):
    return " ".join(str(text).lower().split())


def format_search(query, ranked, seen):
    best = ranked[0][1]
    shown = ", ".join(f"p.{page} ({score / best:.2f}{', seen' if page in seen else ''})" for page, score in ranked[:SEARCH_K])
    return f'Results for "{query}", best first (score relative to the best): {shown}.'


print(f"Agent contract {AGENT_PROMPT_VERSION}: sha256 {AGENT_CONTRACT_SHA256[:12]}…; tools {TOOL_NAMES}; "
      f"budget {MAX_CALLS} calls, {MAX_PAGES} pages, {MAX_READ} per read, search returns {SEARCH_K}.")

Agent contract e10-agent-v1: sha256 de749414e400…; tools ['search_pages', 'read_pages', 'finish']; budget 6 calls, 12 pages, 4 per read, search returns 8.


In [10]:
# The agent loop, written out: call the model -> run the one tool it chose -> append the result -> repeat, under hard limits.
# `create` is any callable with the signature of openai.OpenAI().chat.completions.create; `spend_guard(upper)` raises if the
# next call's worst-case cost would break the cumulative ceiling; `on_call(record)` receives one log record per model call.
def tool_call_dict(call):
    return {"id": call.id, "type": "function", "function": {"name": call.function.name, "arguments": call.function.arguments}}


def run_episode(create, qid, *, spend_guard=lambda upper: None, on_call=lambda record: None, search=None):
    doc_id = q.at[qid, "doc_id"]
    n_pages = len(DOC_PAGES[doc_id])
    search = search or (lambda text: rank_pages(doc_id, encoder(text)))
    seen = list(q.at[qid, "pages"][:TOP_K])
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, first_message(qid)]
    searched, steps, episode_upper, episode_cost = {}, [], 0.0, 0.0
    final = {"answer": "", "cited_pages": [], "evidence": [], "parse_error": "no finish call"}
    for call_index in range(1, MAX_CALLS + 1):
        forced = call_index == MAX_CALLS
        request = {**REQUEST_PARAMS, "messages": messages, "tools": TOOLS, "tool_choice": FORCE_FINISH if forced else "required"}
        logged = loggable(request)
        bound = agent_prompt_bound(logged)
        upper = upper_cost(logged, bound)
        if not forced and episode_upper + upper + upper_cost({**logged, "tool_choice": FORCE_FINISH}, bound + 2_000) > EPISODE_CAP_USD:
            forced = True  # not enough per-episode budget for this step AND a later finish: finish now
            request["tool_choice"] = FORCE_FINISH
            logged = loggable(request)
        assert bound + MAX_OUTPUT_TOKENS <= CONTEXT_TOKENS, f"{qid}: request could exceed the context window"
        spend_guard(upper)
        record = {"qid": qid, "call_index": call_index, "forced_finish": forced, "fingerprint": fingerprint(request),
                  "prompt_bound": bound, "prompt_estimate": agent_prompt_estimate(logged), "cost_upper_usd": upper,
                  "request": logged, "started_at": utc_now()}
        started = time.perf_counter()
        try:
            response = create(**request)
        except Exception as error:
            record.update(status="api_error", error=f"{type(error).__name__}: {str(error)[:300]}",
                          latency_s=round(time.perf_counter() - started, 3))
            on_call(record)
            raise
        choice = response.choices[0]
        usage = getattr(response, "usage", None)
        details = getattr(usage, "prompt_tokens_details", None) if usage else None
        calls = list(choice.message.tool_calls or [])
        record.update(status="ok", latency_s=round(time.perf_counter() - started, 3), served_model=getattr(response, "model", None),
                      system_fingerprint=getattr(response, "system_fingerprint", None), finish_reason=choice.finish_reason,
                      content=choice.message.content, tool_calls=[tool_call_dict(c) for c in calls],
                      prompt_tokens=getattr(usage, "prompt_tokens", None), completion_tokens=getattr(usage, "completion_tokens", None),
                      cached_prompt_tokens=(getattr(details, "cached_tokens", 0) or 0) if details else 0)
        record["list_price_cost_usd"] = observed_cost(record)
        episode_upper += upper
        episode_cost += record["list_price_cost_usd"]

        # Exactly one tool call is the protocol (tool_choice=required, parallel_tool_calls=false). Anything else costs the step.
        if len(calls) != 1 or choice.finish_reason == "length":
            result = "Protocol error: call exactly one tool." + ("" if forced else " " + budget_line(call_index, seen))
            messages = messages + [{"role": "assistant", "content": choice.message.content or ""},
                                   {"role": "user", "content": result}]
            record.update(action="protocol_error", tool_result=result)
            steps.append({"call": call_index, "action": "protocol_error"})
            on_call(record)
            continue
        call = calls[0]
        name, raw_args = call.function.name, call.function.arguments
        messages = messages + [{"role": "assistant", "content": None, "tool_calls": [tool_call_dict(call)]}]
        images = []
        try:
            args = json.loads(raw_args)
            if not isinstance(args, dict):
                raise ValueError("arguments must be a JSON object")
        except ValueError as error:
            args, name = None, f"invalid:{name}"
            result = f"Error: the arguments are not valid JSON ({error})."
        if name == "finish":
            answer, cited, ev, error = parse_b1_reply(raw_args)
            if error is None or forced:
                final = {"answer": answer, "cited_pages": cited, "evidence": ev, "parse_error": error}
                record.update(action="finish", tool_result=None)
                steps.append({"call": call_index, "action": "finish", "parse_error": error})
                on_call(record)
                break
            result = f"finish was rejected: {error}. Fix it and call finish again."
            steps.append({"call": call_index, "action": "finish_rejected", "error": error})
        elif name == "search_pages":
            text = str(args.get("query", "")).strip()
            key = normalise_query(text)
            if not text or len(text) > 300:
                result = "Error: the query must be 1-300 characters."
                steps.append({"call": call_index, "action": "search_invalid"})
            elif key in searched:
                result = f'Already searched "{text}"; see the earlier result.'
                steps.append({"call": call_index, "action": "search_repeat", "query": text})
            else:
                ranked = search(text)
                searched[key] = [page for page, _ in ranked[:SEARCH_K]]
                result = format_search(text, ranked, seen)
                steps.append({"call": call_index, "action": "search", "query": text, "results": searched[key],
                              "scores": [round(score, 6) for _, score in ranked[:SEARCH_K]]})
        elif name == "read_pages":
            pages = args.get("pages")
            if not isinstance(pages, list) or not pages or not all(isinstance(p, int) and not isinstance(p, bool) for p in pages):
                result = "Error: pages must be a non-empty list of page numbers."
                steps.append({"call": call_index, "action": "read_invalid"})
            else:
                unknown = [p for p in pages if not 1 <= p <= n_pages]
                already = [p for p in pages if p in seen]
                wanted = list(dict.fromkeys(p for p in pages if 1 <= p <= n_pages and p not in seen))
                notes = []
                if len(wanted) > MAX_READ:
                    notes.append(f"only the first {MAX_READ} new pages are shown")
                    wanted = wanted[:MAX_READ]
                room = MAX_PAGES - len(seen)
                if len(wanted) > room:
                    notes.append(f"the page budget allows only {room} more")
                    wanted = wanted[:room]
                if unknown:
                    notes.append(f"pages {unknown} do not exist (the document has {n_pages} pages)")
                if already:
                    notes.append(f"pages {already} are already in view")
                seen += wanted
                images = wanted
                shown = f"Showing pages {wanted} below." if wanted else "No new page to show."
                result = " ".join([shown] + [n[0].upper() + n[1:] + "." for n in notes])
                steps.append({"call": call_index, "action": "read", "requested": pages, "shown": wanted})
        elif not name.startswith("invalid:"):
            result = f"Error: unknown tool {name!r}. Use one of {TOOL_NAMES}."
            steps.append({"call": call_index, "action": "unknown_tool", "name": name})
        else:
            steps.append({"call": call_index, "action": "invalid_arguments"})
        result = result + " " + budget_line(call_index, seen)
        messages = messages + [{"role": "tool", "tool_call_id": call.id, "content": result}]
        if images:  # Chat Completions tool messages carry text only: the images follow in a user message
            messages = messages + [{"role": "user", "content": page_parts(doc_id, images)}]
        record.update(action=steps[-1]["action"], tool_result=result, pages_shown=images)
        on_call(record)
    return {"qid": qid, "answer": final["answer"], "cited_pages": final["cited_pages"], "evidence": final["evidence"],
            "parse_error": final["parse_error"], "pages_seen": seen, "calls": len([s for s in steps]),
            "searches": sum(s["action"] == "search" for s in steps), "steps": steps,
            "forced_finish": any(s["action"] == "finish" for s in steps) and record["forced_finish"],
            "cost_upper_usd": episode_upper, "list_price_cost_usd": episode_cost}


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def observed_cost(record):
    if record.get("completion_tokens") is None:
        return 0.0
    prompt, cached = record.get("prompt_tokens") or 0, record.get("cached_prompt_tokens") or 0
    return max(0, prompt - cached) * PRICE["input"] + cached * PRICE["cached_input"] + record["completion_tokens"] * PRICE["output"]


print("run_episode defined: warm start, one tool per call, guards for repeats, invalid arguments, page budget and unknown tools;"
      " forced finish on the last call or when the per-episode ceiling would be reached.")

run_episode defined: warm start, one tool per call, guards for repeats, invalid arguments, page budget and unknown tools; forced finish on the last call or when the per-episode ceiling would be reached.


In [11]:
# Interface tests of the loop with a SCRIPTED fake model (no API, no encoder). These test the code around the model, never the model:
# nothing a fake returns is a project result. The fake bills the exact text+image estimate, so the budget bound is checked too.
class ScriptedCreate:
    """Returns the scripted tool calls in order; when the request forces finish, it returns the scripted finish (or a default)."""

    def __init__(self, script, *, on_forced=None):
        self.script, self.on_forced, self.requests = list(script), on_forced, []

    def __call__(self, **request):
        self.requests.append(request)
        forced = request["tool_choice"] == FORCE_FINISH
        if forced:
            name, args = self.on_forced or ("finish", {"answer": NOT_ANSWERABLE, "cited_pages": [], "evidence": []})
        else:
            step = self.script.pop(0)
            if step is None:  # a reply with no tool call
                return self._response(None, None, content="I think the answer is 42.")
            name, args = step
        return self._response(name, args, request=request)

    def _response(self, name, args, content=None, request=None):
        calls = None if name is None else [SimpleNamespace(id=f"call_{len(self.requests)}", type="function",
                                                            function=SimpleNamespace(name=name, arguments=args if isinstance(args, str) else json.dumps(args)))]
        prompt = agent_prompt_estimate(loggable(request)) if request else 100
        usage = SimpleNamespace(prompt_tokens=prompt, completion_tokens=40, prompt_tokens_details=SimpleNamespace(cached_tokens=0))
        return SimpleNamespace(model=MODEL, system_fingerprint=None, usage=usage,
                               choices=[SimpleNamespace(finish_reason="tool_calls" if calls else "stop",
                                                        message=SimpleNamespace(content=content, tool_calls=calls))])


TEST_QID = QIDS[0]
TEST_DOC = q.at[TEST_QID, "doc_id"]
warm = q.at[TEST_QID, "pages"][:TOP_K]
others = [p for p in DOC_PAGES[TEST_DOC] if p not in warm]
fake_search = lambda text: [(p, 10.0 - i * 0.1) for i, p in enumerate(others + warm)]  # a fixed ranking: no encoder needed
done = lambda pages: ("finish", {"answer": "x", "cited_pages": pages, "evidence": [{"page": pages[0], "quote": "x"}]})
logged_calls = []

# 1. The normal path: search, read, finish. The cost bound is never below the billed (fake) cost.
fake = ScriptedCreate([("search_pages", {"query": "revenue table"}), ("read_pages", {"pages": others[:2]}), done(others[:1])])
ep = run_episode(fake, TEST_QID, search=fake_search, on_call=logged_calls.append)
assert ep["calls"] == 3 and ep["searches"] == 1 and ep["pages_seen"] == warm + others[:2] and ep["parse_error"] is None
assert all(r["cost_upper_usd"] >= r["list_price_cost_usd"] for r in logged_calls) and len(logged_calls) == 3
assert fake.requests[2]["messages"][-1]["role"] == "user" and fake.requests[2]["messages"][-2]["role"] == "tool"  # images after tool text
assert all(r["tool_choice"] == "required" for r in fake.requests)

# 2. Finish at once: one call, the warm start only.
ep = run_episode(ScriptedCreate([done(warm[:1])]), TEST_QID, search=fake_search)
assert ep["calls"] == 1 and ep["pages_seen"] == warm

# 3. A model that loops on the same search: repeats are refused, and call 6 forces finish.
fake = ScriptedCreate([("search_pages", {"query": "Revenue  "})] + [("search_pages", {"query": "revenue"})] * 4)
ep = run_episode(fake, TEST_QID, search=fake_search)
assert ep["calls"] == MAX_CALLS and ep["searches"] == 1 and ep["forced_finish"] and ep["answer"] == NOT_ANSWERABLE
assert [s["action"] for s in ep["steps"]].count("search_repeat") == 4 and fake.requests[-1]["tool_choice"] == FORCE_FINISH

# 4. Malformed arguments, an unknown tool, a reply with no tool call, nonexistent pages: each costs a step and is explained.
fake = ScriptedCreate([("read_pages", '{"pages": "seven"'), ("zoom", {"page": 3}), None, ("read_pages", {"pages": [0, 10_000]}),
                       done(warm[:1])])
ep = run_episode(fake, TEST_QID, search=fake_search)
assert [s["action"] for s in ep["steps"]] == ["invalid_arguments", "unknown_tool", "protocol_error", "read", "finish"]
assert "do not exist" in fake.requests[4]["messages"][-1]["content"]

# 5. The page budget: 4 warm + 4 + 4 = 12; a further read shows nothing; more than 4 requested at once are cut to 4.
fake = ScriptedCreate([("read_pages", {"pages": others[:6]}), ("read_pages", {"pages": others[4:8]}), ("read_pages", {"pages": others[8:10]}),
                       done(others[:1])])
ep = run_episode(fake, TEST_QID, search=fake_search)
assert len(ep["pages_seen"]) == MAX_PAGES and ep["steps"][0]["shown"] == others[:4] and ep["steps"][2]["shown"] == []

# 6. An invalid finish (an abstention with citations) is rejected and can be fixed; on the forced call it is kept as an error.
fake = ScriptedCreate([("finish", {"answer": NOT_ANSWERABLE, "cited_pages": [warm[0]], "evidence": []}), done(warm[:1])])
ep = run_episode(fake, TEST_QID, search=fake_search)
assert [s["action"] for s in ep["steps"]] == ["finish_rejected", "finish"] and ep["parse_error"] is None
bad = ("finish", {"answer": NOT_ANSWERABLE, "cited_pages": [1], "evidence": []})
ep = run_episode(ScriptedCreate([("search_pages", {"query": f"q{i}"}) for i in range(5)], on_forced=bad), TEST_QID, search=fake_search)
assert ep["forced_finish"] and ep["parse_error"] is not None and ep["calls"] == MAX_CALLS

# 7. The cumulative guard stops before sending, and an API failure propagates after being logged.
def refuse(upper):
    raise ValueError("ceiling")
fake = ScriptedCreate([done(warm[:1])])
try:
    run_episode(fake, TEST_QID, search=fake_search, spend_guard=refuse)
    raise AssertionError("the guard must refuse")
except ValueError:
    assert fake.requests == []
failing_log = []
def failing(**request):
    raise TimeoutError("mock transport interruption")
try:
    run_episode(failing, TEST_QID, search=fake_search, on_call=failing_log.append)
    raise AssertionError("an API failure must propagate")
except TimeoutError:
    assert failing_log[-1]["status"] == "api_error"

# 8. Worst case: 6 calls with 12 pages stays under the per-episode ceiling and inside the context window.
fake = ScriptedCreate([("read_pages", {"pages": others[i:i + 4]}) for i in (0, 4)] + [("search_pages", {"query": f"q{i}"}) for i in range(3)])
worst_log = []
ep = run_episode(fake, TEST_QID, search=fake_search, on_call=worst_log.append)
assert ep["calls"] == MAX_CALLS and len(ep["pages_seen"]) == MAX_PAGES and ep["cost_upper_usd"] <= EPISODE_CAP_USD
print(f"Agent loop interface tests passed (8 scripted scenarios). Worst-case episode bound on {TEST_QID}: "
      f"${ep['cost_upper_usd']:.5f} (cap ${EPISODE_CAP_USD}), largest prompt bound {max(r['prompt_bound'] for r in worst_log):,} tokens.")

Agent loop interface tests passed (8 scripted scenarios). Worst-case episode bound on mmlb-0042: $0.01079 (cap $0.03), largest prompt bound 16,911 tokens.


## Stage 3–4 — plans, the fail-closed executor and the paid gate (approved 2026-11-27, cumulative ceiling $3.50)

In [12]:
# The smoke plan (6 agent episodes, not scored) and the full plan (118 blocks of B1s, AGa, AGb, order counterbalanced), seeded.
from itertools import permutations

smoke_qids = sorted(np.random.default_rng(SEED + 1).choice(QIDS, size=6, replace=False).tolist())
SMOKE_PLAN = [{"execution_index": i + 1, "replicate_id": "smoke", "block_index": None, "position_in_block": None, "qid": qid, "arm": "AG"}
              for i, qid in enumerate(smoke_qids)]

order_rng = np.random.default_rng(SEED + 2)
order = list(QIDS)
order_rng.shuffle(order)
ORDERS = list(permutations(["B1s", "AGa", "AGb"]))
assigned = [ORDERS[i % len(ORDERS)] for i in range(len(order))]
order_rng.shuffle(assigned)
FULL_PLAN = []
for block, (qid, arms) in enumerate(zip(order, assigned), start=1):
    for position, arm in enumerate(arms, start=1):
        FULL_PLAN.append({"execution_index": len(FULL_PLAN) + 1, "replicate_id": {"B1s": "control", "AGa": "agent_r1", "AGb": "agent_r2"}[arm],
                          "block_index": block, "position_in_block": position, "qid": qid, "arm": arm})
assert Counter(item["arm"] for item in FULL_PLAN) == {"B1s": 118, "AGa": 118, "AGb": 118}
first = Counter(item["arm"] for item in FULL_PLAN if item["position_in_block"] == 1)
assert max(first.values()) - min(first.values()) <= 2, first


def plan_sha256(plan):
    return sha256_bytes(json.dumps(plan, sort_keys=True).encode())


b1_upper = sum(B1_REQUESTS[item["qid"]]["upper_usd"] for item in FULL_PLAN if item["arm"] == "B1s")
worst_episode = ep["cost_upper_usd"]  # the 12-page, 6-call scripted worst case above (one document; others differ by page sizes)
plan_table = pd.DataFrame([
    {"plan": "smoke", "items": len(SMOKE_PLAN), "worst-case USD": round(6 * worst_episode, 4), "sha256": plan_sha256(SMOKE_PLAN)[:12]},
    {"plan": "full", "items": len(FULL_PLAN), "worst-case USD": round(b1_upper + 236 * worst_episode, 4), "sha256": plan_sha256(FULL_PLAN)[:12]},
])
display(plan_table)
print(f"Smoke questions: {smoke_qids}. First position per arm: {dict(first)}.")
print(f"The full plan's worst case (${b1_upper + 236 * worst_episode:.2f}) assumes every episode uses all 6 calls and 12 pages; "
      f"the cumulative ceiling is ${BUDGET_USD:.2f}. The executor checks each call's bound before sending and stops fail-closed "
      "if the next call could cross the ceiling, so the ceiling holds even then.")
(OUT_DIR / "plan_identity.json").write_text(json.dumps({
    "smoke_sha256": plan_sha256(SMOKE_PLAN), "full_sha256": plan_sha256(FULL_PLAN), "agent_contract_sha256": AGENT_CONTRACT_SHA256,
    "smoke_qids": smoke_qids, "worst_episode_usd": worst_episode, "full_worst_case_usd": b1_upper + 236 * worst_episode}, indent=2) + "\n")

,plan,items,worst-case USD,sha256
0,smoke,6,0.0648,2690f1859dcc
1,full,354,2.6410,edd9a6cc8890


Smoke questions: ['mmlb-0045', 'mmlb-0999', 'mmlb-1000', 'mmlb-1002', 'mmlb-1051', 'mmlb-1061']. First position per arm: {'AGb': 38, 'B1s': 40, 'AGa': 40}.
The full plan's worst case ($2.64) assumes every episode uses all 6 calls and 12 pages; the cumulative ceiling is $3.50. The executor checks each call's bound before sending and stops fail-closed if the next call could cross the ceiling, so the ceiling holds even then.


483

In [13]:
# The fail-closed executor: resumable by execution index, at most two attempts per item, an append-only call log and item log,
# a cumulative ceiling checked before every call (prior runs included), and a stop if the served model or fingerprint changes.
def run_b1(create, qid, *, spend_guard, on_call):
    """One B1s call: the frozen E8 L-B1 request, byte for byte."""
    request = build_b1_request(qid)
    assert fingerprint(request) == B1_REQUESTS[qid]["fingerprint"], f"{qid}: B1s request changed after planning"
    upper = B1_REQUESTS[qid]["upper_usd"]
    spend_guard(upper)
    record = {"qid": qid, "call_index": 1, "forced_finish": False, "fingerprint": B1_REQUESTS[qid]["fingerprint"],
              "prompt_bound": B1_REQUESTS[qid]["prompt_tokens"], "prompt_estimate": B1_REQUESTS[qid]["prompt_tokens"],
              "cost_upper_usd": upper, "request": B1_REQUESTS[qid]["loggable"], "started_at": utc_now()}
    started = time.perf_counter()
    try:
        response = create(**request)
    except Exception as error:
        record.update(status="api_error", error=f"{type(error).__name__}: {str(error)[:300]}", latency_s=round(time.perf_counter() - started, 3))
        on_call(record)
        raise
    choice = response.choices[0]
    raw = choice.message.content or ""
    answer, cited, ev, error = parse_b1_reply(raw)
    if choice.finish_reason != "stop" and error is None:
        error = f"reply did not finish normally (finish_reason={choice.finish_reason!r})"
    usage = getattr(response, "usage", None)
    details = getattr(usage, "prompt_tokens_details", None) if usage else None
    record.update(status="ok", latency_s=round(time.perf_counter() - started, 3), served_model=getattr(response, "model", None),
                  system_fingerprint=getattr(response, "system_fingerprint", None), finish_reason=choice.finish_reason, content=raw,
                  tool_calls=[], action="answer", prompt_tokens=getattr(usage, "prompt_tokens", None),
                  completion_tokens=getattr(usage, "completion_tokens", None),
                  cached_prompt_tokens=(getattr(details, "cached_tokens", 0) or 0) if details else 0)
    record["list_price_cost_usd"] = observed_cost(record)
    on_call(record)
    pages = q.at[qid, "pages"][:TOP_K]
    return {"qid": qid, "answer": answer, "cited_pages": cited, "evidence": ev, "parse_error": error, "pages_seen": list(pages),
            "calls": 1, "searches": 0, "steps": [{"call": 1, "action": "answer"}], "forced_finish": False,
            "cost_upper_usd": upper, "list_price_cost_usd": record["list_price_cost_usd"]}


def budget_charge(record):
    """A failed call can still be billed: without returned usage, charge its precomputed upper bound."""
    return observed_cost(record) if record.get("completion_tokens") is not None else record.get("cost_upper_usd", 0.0)


def prior_spend(exclude=None):
    """Budget charged by every earlier E10 run (smoke included), so the ceiling is cumulative."""
    total = 0.0
    for log in sorted(RUNS_DIR.glob("*/calls.jsonl")):
        if exclude is None or log.parent != Path(exclude):
            total += sum(budget_charge(r) for r in read_jsonl(log))
    return total


def execute(create, run_dir, plan, *, budget_usd, prior_usd=0.0, stop_after=None, search=None):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    calls_path, items_path = run_dir / "calls.jsonl", run_dir / "items.jsonl"
    calls = read_jsonl(calls_path) if calls_path.is_file() else []
    items = read_jsonl(items_path) if items_path.is_file() else []
    by_index = {item["execution_index"]: item for item in plan}
    assert all(r["execution_index"] in by_index for r in calls + items), "the logs contain an item that is not in this plan"
    done = {r["execution_index"]: r for r in items if r["status"] == "ok"}
    attempts = {index: len({r["attempt_id"] for r in calls if r["execution_index"] == index}) for index in by_index}
    state = {"spent": prior_usd + sum(budget_charge(r) for r in calls),
             "models": {r["served_model"] for r in calls if r.get("status") == "ok"},
             "fingerprints": {r["system_fingerprint"] for r in calls if r.get("status") == "ok"}}
    finished_now = 0
    for item in plan:
        index = item["execution_index"]
        if index in done:
            continue
        if attempts[index] >= 2:
            raise RuntimeError(f"execution index {index} exhausted its two attempts")
        attempt_id = f"{index}-{attempts[index] + 1}"

        def guard(upper):
            if state["spent"] + upper > budget_usd:
                raise ValueError(f"the next call could exceed the cumulative E10 ceiling: ${state['spent'] + upper:.6f} > ${budget_usd:.2f}")

        def on_call(record):
            record = {**item, "attempt_id": attempt_id, **record}
            if record["status"] == "ok":
                models, prints = state["models"] | {record["served_model"]}, state["fingerprints"] | {record["system_fingerprint"]}
                if len(models) != 1 or len(prints) != 1:
                    record["status"] = "identity_mismatch"
            with calls_path.open("a") as handle:
                handle.write(json.dumps(record) + "\n")
            state["spent"] += budget_charge(record)
            if record["status"] == "identity_mismatch":
                raise RuntimeError(f"served model or system fingerprint changed: {record['served_model']}, {record['system_fingerprint']}")
            if record["status"] == "ok":
                state["models"].add(record["served_model"])
                state["fingerprints"].add(record["system_fingerprint"])

        attempts[index] += 1
        try:
            if item["arm"] == "B1s":
                result = run_b1(create, item["qid"], spend_guard=guard, on_call=on_call)
            else:
                result = run_episode(create, item["qid"], spend_guard=guard, on_call=on_call, search=search)
        except Exception as error:
            with items_path.open("a") as handle:
                handle.write(json.dumps({**item, "attempt_id": attempt_id, "status": "interrupted", "completed_at": utc_now(),
                                         "error": f"{type(error).__name__}: {str(error)[:300]}"}) + "\n")
            raise RuntimeError(f"E10 stopped at execution index {index} ({type(error).__name__}); resume is explicit") from error
        with items_path.open("a") as handle:
            handle.write(json.dumps({**item, "attempt_id": attempt_id, "status": "ok", "completed_at": utc_now(), **result}) + "\n")
        done[index] = result
        finished_now += 1
        if stop_after is not None and finished_now >= stop_after:
            break
    run_calls = read_jsonl(calls_path) if calls_path.is_file() else []
    summary = {"complete": len(done) == len(plan), "items_done": len(done), "items_planned": len(plan), "calls": len(run_calls),
               "observed_list_price_cost_usd": round(sum(observed_cost(r) for r in run_calls), 9),
               "budget_accounted_usd_including_prior": round(state["spent"], 9), "served_models": sorted(map(str, state["models"])),
               "system_fingerprints": sorted(map(str, state["fingerprints"])), "updated_at": utc_now()}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary


# Interface tests with the scripted fake: resume, interruption (charged at its bound), cumulative ceiling, model change.
class FakeB1:
    def __call__(self, **request):
        usage = SimpleNamespace(prompt_tokens=b1_prompt_tokens(loggable(request)), completion_tokens=12,
                                prompt_tokens_details=SimpleNamespace(cached_tokens=0))
        raw = json.dumps({"answer": NOT_ANSWERABLE, "cited_pages": [], "evidence": []})
        return SimpleNamespace(model=MODEL, system_fingerprint=None, usage=usage,
                               choices=[SimpleNamespace(finish_reason="stop", message=SimpleNamespace(content=raw, tool_calls=None))])


class FakeBoth:
    """B1s requests (response_format) get a B1 reply; agent requests get finish at once. Optional failure or model change."""
    def __init__(self, fail_on=None, model_change_on=None):
        self.calls, self.fail_on, self.model_change_on = 0, fail_on, model_change_on

    def __call__(self, **request):
        self.calls += 1
        if self.calls == self.fail_on:
            raise TimeoutError("mock transport interruption")
        if "tools" not in request:
            response = FakeB1()(**request)
        else:
            response = ScriptedCreate([("finish", {"answer": NOT_ANSWERABLE, "cited_pages": [], "evidence": []})])(**request)
        if self.calls == self.model_change_on:
            response.model = MODEL + "-changed"
        return response


mock_plan = FULL_PLAN[:9]
with tempfile.TemporaryDirectory() as tmp:
    fake = FakeBoth()
    partial = execute(fake, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD, stop_after=4, search=fake_search)
    assert not partial["complete"] and partial["items_done"] == 4
    complete = execute(fake, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD, search=fake_search)
    assert complete["complete"] and fake.calls == 9 and complete["calls"] == 9
    try:
        execute(FakeBoth(fail_on=2), Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD, search=fake_search)
        raise AssertionError("an API failure must stop the run")
    except RuntimeError as err:
        assert "stopped" in str(err)
    resumed = execute(FakeBoth(), Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD, search=fake_search)
    assert resumed["complete"] and resumed["calls"] == 10
    assert resumed["budget_accounted_usd_including_prior"] > resumed["observed_list_price_cost_usd"]  # the failure is charged its bound
    refused = FakeBoth()
    try:
        execute(refused, Path(tmp) / "c", mock_plan, budget_usd=1e-9, search=fake_search)
        raise AssertionError("the ceiling must refuse")
    except RuntimeError as err:
        assert refused.calls == 0 and isinstance(err.__cause__, ValueError)
    try:
        execute(FakeBoth(), Path(tmp) / "d", mock_plan, budget_usd=BUDGET_USD, prior_usd=BUDGET_USD, search=fake_search)
        raise AssertionError("prior spend must count")
    except RuntimeError as err:
        assert isinstance(err.__cause__, ValueError)
    try:
        execute(FakeBoth(model_change_on=3), Path(tmp) / "e", mock_plan, budget_usd=BUDGET_USD, search=fake_search)
        raise AssertionError("a served-model change must stop the run")
    except RuntimeError as err:
        assert "changed" in str(err.__cause__)
print("Executor interface tests passed with fakes: resume, API failure (charged at its bound) and explicit resume, cumulative and "
      "prior-spend ceiling, served-model change.")

Executor interface tests passed with fakes: resume, API failure (charged at its bound) and explicit resume, cumulative and prior-spend ceiling, served-model change.


In [14]:
# The paid gate. offline: nothing is sent. smoke: the 6 smoke episodes. full: the 354 planned items (resumable).
def notebook_code_sha256(path):
    cells = json.loads(Path(path).read_text())["cells"]
    return sha256_bytes("\n\n".join("".join(c["source"]) for c in cells if c["cell_type"] == "code").encode())


NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "10_agentic_reader.ipynb"
RUN = None
if MODE == "offline":
    print("Offline: no request is sent. Set VISUALDOC_E10_MODE=smoke or full (with the frozen spec and approval) to run.")
else:
    assert SPEC is not None and SPEC_FREEZE_COMMIT, "paid modes need the committed E10 specification"
    assert SPEC["agent"]["agent_contract_sha256"] == AGENT_CONTRACT_SHA256, "the agent contract differs from the frozen spec"
    assert SPEC["plans"]["smoke_sha256"] == plan_sha256(SMOKE_PLAN) and SPEC["plans"]["full_sha256"] == plan_sha256(FULL_PLAN)
    assert SPEC["budget"]["cumulative_ceiling_usd"] == BUDGET_USD and PARITY_PASSED
    committed_nb = subprocess.run(["git", "show", f"HEAD:{NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}"], cwd=PROJECT_ROOT,
                                  capture_output=True, check=True).stdout
    with tempfile.NamedTemporaryFile("wb", suffix=".ipynb", delete=False) as handle:
        handle.write(committed_nb)
    assert notebook_code_sha256(handle.name) == notebook_code_sha256(NOTEBOOK_PATH), "the notebook code differs from HEAD: commit it first"
    assert os.environ.get("OPENAI_API_KEY"), "export OPENAI_API_KEY in the shell (the notebook never reads .env)"
    from openai import OpenAI
    # The Datalab exports OPENAI_BASE_URL as an empty string, which the SDK would use as the URL: fall back to the official endpoint.
    API_BASE_URL = os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1"
    client = OpenAI(max_retries=0, timeout=120, base_url=API_BASE_URL)
    if encoder.model is None:
        encoder._load()
    resume = os.environ.get("VISUALDOC_E10_RUN", "").strip()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    RUN = Path(resume) if resume else RUNS_DIR / f"{MODE}_{stamp}_{uuid.uuid4().hex[:8]}"
    plan = SMOKE_PLAN if MODE == "smoke" else FULL_PLAN
    settings = {"mode": MODE, "plan": plan, "plan_sha256": plan_sha256(plan), "spec_sha256": SPEC_SHA256,
                "spec_commit": SPEC_FREEZE_COMMIT, "agent_contract_sha256": AGENT_CONTRACT_SHA256,
                "notebook_code_sha256": notebook_code_sha256(NOTEBOOK_PATH), "api_base_url": API_BASE_URL, "encoder": {**E5_IDENTITY, "device": "cpu", "threads": encoder.threads}}
    RUN.mkdir(parents=True, exist_ok=True)
    if (RUN / "settings.json").is_file():
        assert json.loads((RUN / "settings.json").read_text()) == settings, "resuming a run with different settings"
    else:
        (RUN / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
    prior = prior_spend(exclude=RUN)
    print(f"{MODE}: {len(plan)} items -> {RUN.relative_to(PROJECT_ROOT)}; prior E10 spend ${prior:.6f}, ceiling ${BUDGET_USD:.2f}.")
    summary = execute(client.chat.completions.create, RUN, plan, budget_usd=BUDGET_USD, prior_usd=prior)
    display(pd.Series(summary, name=RUN.name).to_frame())

Offline: no request is sent. Set VISUALDOC_E10_MODE=smoke or full (with the frozen spec and approval) to run.


## Stage 5 — evaluation and the frozen decision rules

In [15]:
# Names the verbatim scorer cells expect (as in notebook 07's and 08's setup). `dev` (the 118 questions) is defined above.
import ast
import inspect
import re
from math import isclose

NB03_PATH = PROJECT_ROOT / "notebooks" / "03_evaluation.ipynb"
assert NB03_PATH.is_file(), "notebook 03 is tracked in git: restore it"
print(f"Scoring the {len(dev)} development questions of the 14 held-out-14 documents.")

Scoring the 118 development questions of the 14 held-out-14 documents.


In [16]:
# S-v1: the frozen MMLongBench-Doc scorer (notebook 03, cell f653d232), written out unchanged. Correct = score > 0; strict = score == 1.
def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            new_distances.append(distances[i1] if c1 == c2 else 1 + min(distances[i1], distances[i1 + 1], new_distances[-1]))
        distances = new_distances
    return distances[-1]


def anls_compute(groundtruth, prediction, threshold=0.5):
    length = max(len(groundtruth.upper()), len(prediction.upper()))
    anls = 1.0 - (0.0 if length == 0 else levenshtein_distance(groundtruth, prediction) / length)
    return 0.0 if anls <= threshold else anls


def is_float_equal(reference, prediction, include_percentage=False, is_close=False):
    def get_precision(value):
        text = str(value)
        return len(text.split(".")[-1]) if "." in text else 3

    reference = float(str(reference).strip().rstrip("%").strip())
    try:
        prediction = float(str(prediction).strip().rstrip("%").strip())
    except (TypeError, ValueError):
        return False
    for item in ([reference / 100, reference, reference * 100] if include_percentage else [reference]):
        if is_close and isclose(item, prediction, rel_tol=0.01):
            return True
        precision = max(min(get_precision(prediction), get_precision(item)), 2)
        if round(prediction, precision) == round(item, precision):
            return True
    return False


def get_clean_string(s):
    s = str(s).lower().strip()
    # Official quirk kept: the code calls s.rstrip("mile") etc. without assigning the result, so unit suffixes stay.
    s = re.sub(r"\s*\([^)]*\)", "", s).strip()  # remove parenthesised text
    s = re.sub(r"^['\"]|['\"]$", "", s).strip()  # remove surrounding quotes
    s = s.strip().lstrip("$").strip()
    return s.strip().rstrip("%").strip()


EXACT_MATCH_PATTERNS = [
    r"\b\d+(-\d+|\s\d+)?\b",  # phone-like numbers
    r"\b\d{4}[-\s]\d{2}[-\s]\d{2}\b",  # YYYY-MM-DD
    r"\b\d{4}[-\s]\d{2}\b",  # YYYY-MM
    r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # e-mail address
]


def is_exact_match(s):
    return ("https://" in s or s.endswith(".py") or s.endswith("ipynb") or s.startswith("page") or "a.m." in s or "p.m." in s
            or any(re.fullmatch(pattern, s) for pattern in EXACT_MATCH_PATTERNS))


def isfloat(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def as_list(value):
    if isinstance(value, str) and value.startswith("["):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass  # deviation: the official eval() would raise here
    return value if isinstance(value, list) else [value]


def eval_score(gt, pred, answer_type):
    if answer_type == "Int":
        try:
            gt, pred = int(gt), int(float(pred))
        except Exception:  # the official code uses a bare except
            pred = ""
        score = gt == pred
    elif answer_type == "Float":
        try:
            gt, pred = float(get_clean_string(str(gt))), float(get_clean_string(str(pred)))
        except Exception:
            pred = ""
        score = is_float_equal(gt, pred, include_percentage=True, is_close=True)
    elif answer_type in ("Str", "None"):
        gt, pred = get_clean_string(gt), get_clean_string(pred)
        score = gt == pred if is_exact_match(gt) else anls_compute(gt, pred)
    else:  # List
        gt, pred = as_list(gt), as_list(pred)
        if len(gt) != len(pred):
            score = 0.0
        else:
            gt, pred = sorted(get_clean_string(i) for i in gt), sorted(get_clean_string(i) for i in pred)
            score = "-".join(gt) == "-".join(pred) if isfloat(gt[0]) or is_exact_match(gt[0]) else min(anls_compute(g, p) for g, p in zip(gt, pred))
    return float(score)


# 1. Unchanged: every function above has the same source as the frozen cell, and so does the pattern list.
frozen_cell = next("".join(c["source"]) for c in json.loads(NB03_PATH.read_text())["cells"] if c.get("id") == "f653d232")
frozen_tree = ast.parse(frozen_cell)
frozen_functions = {node.name: ast.get_source_segment(frozen_cell, node) for node in frozen_tree.body if isinstance(node, ast.FunctionDef)}
SCORER_FUNCTIONS = ["levenshtein_distance", "anls_compute", "is_float_equal", "get_clean_string", "is_exact_match", "isfloat", "as_list", "eval_score"]
assert set(SCORER_FUNCTIONS) <= set(frozen_functions)
assert all(inspect.getsource(globals()[name]).strip() == frozen_functions[name].strip() for name in SCORER_FUNCTIONS)
frozen_patterns = next(ast.literal_eval(node.value) for node in frozen_tree.body
                       if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", None) == "EXACT_MATCH_PATTERNS")
assert frozen_patterns == EXACT_MATCH_PATTERNS

# 2. The 17 hand-traced cases of notebook 03.
SCORING_CASES = [  # reference, prediction, format, expected
    ("Less well-off", "less well off", "Str", 1 - 1 / 13), ("Pew Research Center", "Pew Research Centre", "Str", 1 - 2 / 19),
    ("Latinos", "Asian Americans", "Str", 0.0), ("Not answerable", "Not answerable", "None", 1.0), ("Not answerable", "The report does not say", "None", 0.0),
    ("12", "12.0", "Int", 1.0), ("12", "twelve", "Int", 0.0), ("18.29%", "18.3", "Float", 1.0), ("0.1", "10%", "Float", 1.0), ("51%", "49", "Float", 0.0),
    ("['Circle', 'Rectangle']", "['rectangle', 'circle']", "List", 1.0), ("['White', '10%']", "['White']", "List", 0.0),
    ("['2014', '2016']", "['2016', '2014']", "List", 1.0), ("https://www.example.com", "www.example.com", "Str", 0.0), ("2016-06", "2016-06", "Str", 1.0),
    ("$1,200 (approx.)", "1,200", "Str", 1.0), ("5 miles", "5", "Str", 0.0)]
assert len(SCORING_CASES) == 17 and all(isclose(eval_score(r, p, f), e, abs_tol=1e-9) for r, p, f, e in SCORING_CASES)
print(f"S-v1: {len(SCORER_FUNCTIONS)} functions and the pattern list equal notebook 03's frozen cell f653d232; {len(SCORING_CASES)} hand-traced cases pass.")

S-v1: 8 functions and the pattern list equal notebook 03's frozen cell f653d232; 17 hand-traced cases pass.


In [17]:
# S-v2: a deterministic layer over S-v1 that returns (class, reason) with class in {correct, wrong, gray}. Gray goes to blind adjudication.
SCALE_WORDS = {"thousand": 1e3, "k": 1e3, "million": 1e6, "millions": 1e6, "mn": 1e6, "billion": 1e9, "billions": 1e9, "bn": 1e9, "trillion": 1e12}
NUMBER_PATTERN = re.compile(r"(\()?(-?\d[\d,]*(?:\.\d+)?)(\))?\s*(%|(?:percent|thousand|millions?|billions?|trillion|bn|mn|k)\b)?")
NUMERIC_TEXT_RESIDUE = re.compile(r"us\$|usd|eur|gbp|percent|thousand|millions?|billions?|trillion|\bbn\b|\bmn\b|\bk\b|[$€£%(),.\-\s\d]")
POWERS_OF_1000 = (1e-9, 1e-6, 1e-3, 1e3, 1e6, 1e9)  # "m" and "b" are not scale words: metres, bytes


def numbers_in(value):
    """Every number in a reply or reference, after removing currency signs; a scale word multiplies, '(x)' is negative."""
    if isinstance(value, bool):
        return []
    if isinstance(value, (int, float)):
        return [float(value)]
    text = re.sub(r"us\$|usd|eur|gbp|[$€£]", " ", str(value).lower())
    found = []
    for open_paren, digits, close_paren, unit in NUMBER_PATTERN.findall(text):
        number = float(digits.replace(",", ""))
        found.append((-number if open_paren and close_paren else number) * SCALE_WORDS.get(unit, 1.0))
    return found


def numeric_text(value):
    """True when a value is one number with at most currency, %, separators and a scale word around it."""
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return True
    return NUMERIC_TEXT_RESIDUE.sub("", str(value).lower()) == "" and len(numbers_in(value)) == 1


def same_value(reference, prediction, answer_format):
    # Float: within 1 %, including the official x100 / /100 percentage variants; Int: exactly equal.
    if answer_format == "Float":
        return any(isclose(candidate, prediction, rel_tol=0.01, abs_tol=1e-9) for candidate in (reference, reference / 100, reference * 100))
    return isclose(reference, prediction, rel_tol=0.0, abs_tol=1e-9)


def numeric_class(reference, prediction, answer_format):
    """correct / scale / among / wrong / None (the reference is not one number)."""
    ref, pred = numbers_in(reference), numbers_in(prediction)
    if len(ref) != 1 or not pred:
        return None if len(ref) != 1 else "wrong"
    if len(pred) > 1:
        return "among" if any(same_value(ref[0], p, answer_format) for p in pred) else "wrong"
    if same_value(ref[0], pred[0], answer_format):
        return "correct"
    if any(same_value(ref[0] * f, pred[0], answer_format) for f in POWERS_OF_1000):
        return "scale"
    return "wrong"


def normalise(value):
    text = json.dumps(value) if isinstance(value, list) else str(value)
    return " ".join(re.sub(r"[^a-z0-9%.]+", " ", text.lower()).split())


def contains_phrase(haystack, needle):
    return bool(needle) and f" {needle} " in f" {haystack} "


def canonical_item(item):
    """A list item for comparison: a number as a number ('$35,000' = 35000), anything else normalised text."""
    return f"{numbers_in(item)[0]:.6g}" if numeric_text(item) else normalise(item)


def item_set(value):
    return {canonical_item(item) for item in as_list(value)} - {""}


def items_match(a, b):
    """Equal, one a phrase of the other, or ANLS above 0.5 (S-v1's threshold)."""
    return a == b or contains_phrase(a, b) or contains_phrase(b, a) or anls_compute(a, b) > 0


# References the frozen E6 error audit labelled as annotation issues: scored by S-v1 as usual, sent to adjudication by S-v2.
e6_audit = pd.read_csv(FROZEN_DIR / "e7" / "design" / "e6_error_audit.csv")
DEBATABLE_QIDS = set(e6_audit.loc[e6_audit["category"] == "annotation issue", "qid"])


NUMERIC_REASON = {"correct": ("correct", "numeric match after normalisation (currency signs, separators, %, scale words)"),
                  "scale": ("gray", "same number up to a factor of 1,000^n (unit scale ambiguous)"),
                  "among": ("gray", "the reference value is one of several numbers in the reply"),
                  "wrong": ("wrong", "numeric mismatch")}


def score_v2(reference, prediction, answer_format, debatable=False):
    s1 = eval_score(reference, prediction, answer_format)
    is_abstention = isinstance(prediction, str) and prediction.strip().lower() == NOT_ANSWERABLE.lower()
    if answer_format == "None":
        return ("correct", "exact abstention") if s1 >= 1 else ("wrong", "answered an unanswerable question")
    if is_abstention:
        return "wrong", "false abstention"
    if s1 >= 1:
        return "correct", "S-v1 full match"
    if debatable:
        return "gray", "reference flagged as debatable by the frozen E6 audit"
    if answer_format in ("Int", "Float"):
        return NUMERIC_REASON[numeric_class(reference, prediction, answer_format) or "wrong"]
    if answer_format == "Str":
        if numeric_text(reference) and numeric_text(prediction):
            numeric = numeric_class(reference, prediction, "Float")
            if numeric in ("correct", "scale"):
                return NUMERIC_REASON[numeric]
        ref, pred = normalise(reference), normalise(prediction)
        if ref and ref == pred:
            return "correct", "equal after case and punctuation normalisation"
        if 0 < s1 < 1:
            return "gray", "partial ANLS credit (0.5 < ANLS < 1)"
        if contains_phrase(pred, ref):
            return "gray", "the reference is a phrase inside a longer reply"
        if len(pred.split()) >= 2 and contains_phrase(ref, pred):
            return "gray", "the reply is a phrase inside the reference"
        return "wrong", "string mismatch"
    # List: a reply missing a reference item is wrong (the benchmark asks for the whole list); only full matches go to a human.
    ref_items, pred_items = item_set(reference), item_set(prediction)
    if ref_items and ref_items == pred_items:
        return "correct", "same items after normalisation"
    if ref_items and all(any(items_match(r, p) for p in pred_items) for r in ref_items):
        if len(pred_items) > len(ref_items):
            return "gray", "every reference item present, with extra items"
        return "gray", "every reference item matched up to formatting"
    return "wrong", "a reference item is missing"


# Hand checks on invented values, including what S-v2 cannot settle (negation, debatable references): those stay gray or wrong for a human.
V2_CASES = [  # reference, reply, format, expected class
    ("Not answerable", NOT_ANSWERABLE, "None", "correct"), ("Not answerable", "7", "None", "wrong"), ("7", NOT_ANSWERABLE, "Int", "wrong"),
    ("1234", "1,234 units", "Int", "correct"), ("12", "12%", "Int", "correct"), ("15", "between 15 and 20", "Int", "gray"),
    ("3.5", "3.5 million", "Float", "gray"), ("1200", "$1.2 billion", "Float", "gray"), ("42.0", "17.3", "Float", "wrong"),
    ("300 million", "300", "Str", "gray"), ("Northern region", "The northern region had the most visitors", "Str", "gray"),
    ("Special Agent Ana Lee", "Ana Lee", "Str", "gray"), ("Ana Lee", "Bo Chen", "Str", "wrong"),
    ("Paris", "Not Paris but Lyon", "Str", "gray"),  # failure case: containment cannot see the negation; the adjudicator must
    ("Less well-off", "less well off", "Str", "correct"), ("Pew Research Center", "Pew Research Centre", "Str", "gray"),
    ("['Circle', 'Rectangle']", ["rectangle", "circle"], "List", "correct"), ("['$35,000', '$40,684']", [35000, 40684], "List", "correct"),
    ("['Apple Inc', 'Banana']", ["apple", "banana co"], "List", "gray"), ("['a', 'b']", ["a", "b", "c"], "List", "gray"),
    ("['a', 'b', 'c']", ["a", "b"], "List", "wrong"), ("['a']", ["x"], "List", "wrong")]
assert score_v2("12", "13", "Int", debatable=True) == ("gray", "reference flagged as debatable by the frozen E6 audit")
assert score_v2("12", "12", "Int", debatable=True)[0] == "correct"  # S-v1 = 1 stays correct
observed = {(r, str(p), f): score_v2(r, p, f)[0] for r, p, f, _ in V2_CASES}
assert all(observed[(r, str(p), f)] == expected for r, p, f, expected in V2_CASES), {k: v for k, v in observed.items()}
print(f"S-v2: {len(V2_CASES)} invented cases classified as expected (including a negation that only a human can resolve).")

S-v2: 22 invented cases classified as expected (including a negation that only a human can resolve).


In [18]:
# The two scorer cells above are byte-identical to notebook 07's (pinned SHA-256), and therefore to the scoring every frozen verdict used.
SCORER_PINS = {"b1-scorer-v1": "329f91f3ceda29146ec8a0c2467f11fdf7fdbd66c1e6af29123e1983a3fc7ee7",
               "b1-scorer-v2": "902708ba537d580db8c67858fe9a2231e5b546f1538a2b955a3eadff551b8707"}
this_nb = {c.get("id"): "".join(c["source"]) for c in json.loads(NOTEBOOK_PATH.read_text())["cells"]}
nb07_cells = {c.get("id"): "".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks" / "07_baseline_consolidation.ipynb").read_text())["cells"]}
for nb07_id, here_id in (("b1-scorer-v1", "e10-scorer-v1"), ("b1-scorer-v2", "e10-scorer-v2")):
    assert sha256_bytes(this_nb[here_id].encode()) == SCORER_PINS[nb07_id] == sha256_bytes(nb07_cells[nb07_id].encode()), here_id
# Replay check: S-v1 re-scores the frozen E9 B1s replies exactly as the frozen E9 table records them.
e9_table = pd.read_csv(FROZEN_DIR / "e9" / "tables" / "evaluation_per_question.csv")
e9_log = {r["qid"]: r for r in read_jsonl(next((FROZEN_DIR / "e9").glob("run_*_full")) / "execution_log.jsonl")
          if r["arm"] == "B1s" and r["status"] == "ok"}
e9_b1s = e9_table[e9_table["arm"] == "B1s"].set_index("qid")
for qid, record in e9_log.items():
    assert eval_score(dev.at[qid, "answer"], record["parsed"]["answer"], dev.at[qid, "answer_format"]) == e9_b1s.at[qid, "s1_score"], qid
print("Scorer cells equal notebook 07's pinned copies; S-v1 reproduces all 118 frozen E9 B1s scores.")

Scorer cells equal notebook 07's pinned copies; S-v1 reproduces all 118 frozen E9 B1s scores.


In [19]:
# E10 evaluation. It reads a completed full run (the frozen package once it exists), never calls an API, and applies the frozen rules.
requested_eval = os.getenv("VISUALDOC_E10_EVAL_RUN", "").strip()
frozen_runs = sorted(E10_FROZEN.glob("full_*")) if E10_FROZEN.is_dir() else []
live_runs = sorted(p.parent for p in RUNS_DIR.glob("full_*/summary.json") if json.loads(p.read_text()).get("complete"))
EVAL_RUN = PROJECT_ROOT / requested_eval if requested_eval else (frozen_runs[-1] if frozen_runs else (live_runs[-1] if live_runs else None))
E10_EVALUATED = EVAL_RUN is not None
ARMS = ["B1s", "AGa", "AGb"]
if not E10_EVALUATED:
    print("No completed E10 full run exists yet: the evaluation waits for the paid run.")
else:
    settings = json.loads((EVAL_RUN / "settings.json").read_text())
    assert settings["plan"] == FULL_PLAN and settings["plan_sha256"] == plan_sha256(FULL_PLAN), "the run's plan is not this notebook's plan"
    assert settings["agent_contract_sha256"] == AGENT_CONTRACT_SHA256 and (SPEC is None or settings["spec_sha256"] == SPEC_SHA256)
    items = [r for r in read_jsonl(EVAL_RUN / "items.jsonl") if r["status"] == "ok"]
    calls = read_jsonl(EVAL_RUN / "calls.jsonl")
    assert len(items) == 354 and len({r["execution_index"] for r in items}) == 354
    ok_calls = [r for r in calls if r["status"] == "ok"]
    assert {r["served_model"] for r in ok_calls} == {MODEL} and len({r["system_fingerprint"] for r in ok_calls}) == 1
    calls_by_attempt = {}
    for r in calls:
        calls_by_attempt.setdefault(r["attempt_id"], []).append(r)

    rows = []
    for item in items:
        qid, arm = item["qid"], item["arm"]
        reference, fmt = dev.at[qid, "answer"], dev.at[qid, "answer_format"]
        answer, cited, seen = item["answer"], item["cited_pages"], item["pages_seen"]
        own = calls_by_attempt[item["attempt_id"]]
        assert all(c["status"] == "ok" for c in own) and len(own) == item["calls"]
        s1 = eval_score(reference, answer, fmt)
        v2_class, v2_reason = score_v2(reference, answer, fmt, debatable=qid in DEBATABLE_QIDS)
        labelled = bool(q.at[qid, "labelled"])
        row = {"qid": qid, "doc_id": dev.at[qid, "doc_id"], "arm": arm, "answerable": bool(dev.at[qid, "answerable"]),
               "s1_score": s1, "s1_correct": s1 > 0, "v2_class": v2_class, "v2_reason": v2_reason,
               "schema_valid": item["parse_error"] is None,
               "citation_valid": item["parse_error"] is None and set(cited) <= set(seen) and (answer == NOT_ANSWERABLE or bool(cited)),
               "abstained": answer == NOT_ANSWERABLE, "prediction": answer, "pages_seen": len(seen), "calls": item["calls"],
               "searches": item["searches"], "reads": sum(s["action"] == "read" for s in item["steps"]),
               "repeats": sum(s["action"] == "search_repeat" for s in item["steps"]),
               "protocol_errors": sum(s["action"] in ("protocol_error", "invalid_arguments", "unknown_tool", "finish_rejected") for s in item["steps"]),
               "forced_finish": item["forced_finish"], "finished_at_call_1": item["calls"] == 1 and arm != "B1s",
               "prompt_tokens": sum(c["prompt_tokens"] for c in own), "cached_prompt_tokens": sum(c["cached_prompt_tokens"] for c in own),
               "completion_tokens": sum(c["completion_tokens"] for c in own), "cost_usd": sum(c["list_price_cost_usd"] for c in own),
               "latency_s": sum(c["latency_s"] for c in own), "labelled": labelled,
               "incomplete_at_4": bool(q.at[qid, "incomplete_at_4"]), "n_evidence": q.at[qid, "n_evidence"],
               "recall_seen": evidence_recall(seen, q.at[qid, "evidence_pages"]) if labelled else np.nan,
               "complete_seen": (set(q.at[qid, "evidence_pages"]) <= set(seen)) if labelled else np.nan}
        rows.append(row)
    scores = pd.DataFrame(rows)
    assert scores.groupby("arm").size().to_dict() == {"AGa": 118, "AGb": 118, "B1s": 118}

    # The budget bound never undercounted a billed prompt (the pre-call check was safe), and how close the estimate came.
    agent_calls = [c for c in ok_calls if "tools" in c["request"]]
    assert all(c["prompt_bound"] >= c["prompt_tokens"] for c in ok_calls), "a billed prompt exceeded its pre-call bound"
    estimate_gap = pd.Series([c["prompt_tokens"] - c["prompt_estimate"] for c in agent_calls])
    b1_exact = all(c["prompt_tokens"] == c["prompt_estimate"] for c in ok_calls if "tools" not in c["request"])

    # S-v2: reuse an owner label only for an identical blind item; queue the rest for the owner (never a model judgment).
    identity_cols = ["document", "evidence_pages", "question", "answer_format", "reference", "reply"]

    def blind_row(qid, prediction):
        return {"document": dev.at[qid, "doc_id"], "evidence_pages": " ".join(map(str, dev.at[qid, "evidence_pages"])),
                "question": dev.at[qid, "question"], "answer_format": dev.at[qid, "answer_format"],
                "reference": dev.at[qid, "answer"], "reply": json.dumps(prediction) if isinstance(prediction, list) else str(prediction)}

    def blind_identity_sha256(row):
        payload = {column: str(row[column]) for column in identity_cols}
        return sha256_bytes(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode())

    owner_labels = {}
    for package in ("scoring_v2", "e8_scoring_v2", "e9_scoring_v2", "e10_scoring_v2"):
        if (FROZEN_DIR / package / "SHA256SUMS").is_file():
            checks = verify_sha256sums(FROZEN_DIR / package)
            assert checks and all(checks.values()), f"experiments/frozen/{package} differs from its SHA256SUMS"
            labels = pd.read_csv(FROZEN_DIR / package / "adjudication_labels.csv", dtype=str).fillna("")
            owner_labels.update(dict(zip(labels["identity_sha256"], labels["label"])))
    scores["identity"] = [blind_identity_sha256(blind_row(qid, pred)) for qid, pred in zip(scores["qid"], scores["prediction"])]
    scores["v2_final"] = np.where(scores["v2_class"] == "gray", scores["identity"].map(owner_labels).fillna("pending"), scores["v2_class"])
    scores["label_reused"] = (scores["v2_class"] == "gray") & scores["identity"].isin(owner_labels)
    pending = scores[(scores["v2_final"] == "pending") & scores["answerable"]]
    adj = pending.drop_duplicates("identity")[["identity", "qid", "prediction"]].sample(frac=1.0, random_state=SEED + 9).reset_index(drop=True)
    adj["item_id"] = [f"E10-ADJ-{i + 1:03d}" for i in range(len(adj))]
    queue = pd.DataFrame([{"item_id": row.item_id, **blind_row(row.qid, row.prediction)} for row in adj.itertuples()])
    if len(queue):
        queue["label (correct / wrong / unsure)"], queue["note"] = "", ""
    key = adj.merge(pending.groupby("identity")["arm"].agg(lambda a: " | ".join(sorted(set(a)))).rename("arms"), on="identity")
    queue_path, key_path = OUT_DIR / "adjudication_queue.csv", OUT_DIR / "adjudication_key_SEALED.csv"
    if not queue_path.is_file() or pd.read_csv(queue_path, dtype=str).fillna("")["label (correct / wrong / unsure)"].eq("").all():
        queue.to_csv(queue_path, index=False)  # never overwrite a queue the owner has started labelling
    key[["item_id", "qid", "arms", "identity"]].to_csv(key_path, index=False)

    def arm_summary(frame):
        ans, unans, lab = frame[frame["answerable"]], frame[~frame["answerable"]], frame[frame["labelled"]]
        return {"S-v1 correct / 102": int(ans["s1_correct"].sum()), "correct abstentions / 16": int(unans["abstained"].sum()),
                "citation validity": float(frame["citation_valid"].mean()), "schema validity": float(frame["schema_valid"].mean()),
                "S-v2 correct (auto + owner)": int((ans["v2_final"] == "correct").sum()), "S-v2 pending": int((ans["v2_final"] == "pending").sum()),
                "evidence recall of pages seen (101)": float(lab["recall_seen"].mean()), "evidence complete in pages seen (101)": int(lab["complete_seen"].sum()),
                "pages seen (mean)": float(frame["pages_seen"].mean()), "calls (mean)": float(frame["calls"].mean()),
                "searches (mean)": float(frame["searches"].mean()), "prompt tokens (mean)": float(frame["prompt_tokens"].mean()),
                "cached share": float(frame["cached_prompt_tokens"].sum() / frame["prompt_tokens"].sum()),
                "cost USD per question": float(frame["cost_usd"].mean()), "latency s p50": float(frame["latency_s"].median()),
                "latency s p90": float(frame["latency_s"].quantile(0.9))}

    arm_table = pd.DataFrame({arm: arm_summary(scores[scores["arm"] == arm]) for arm in ARMS}).T
    display(arm_table.round(4))

    def per_question(arm, column="s1_correct", subset="answerable"):
        frame = scores[(scores["arm"] == arm) & scores[subset]].set_index("qid").sort_index()
        return frame[column].astype(float)

    def contrast(treatment, control, seed=20261128):
        """treatment/control: per-question values (replicate-averaged where given). Document-cluster statistics (as E9)."""
        diff = (treatment - control).sort_index()
        docs = dev.loc[diff.index, "doc_id"].to_numpy()
        order_docs = sorted(set(docs))
        sums = np.array([diff[docs == d].sum() for d in order_docs])
        counts = np.array([(docs == d).sum() for d in order_docs])
        picks = np.random.default_rng(seed).integers(0, len(order_docs), size=(10_000, len(order_docs)))
        boot = sums[picks].sum(axis=1) / counts[picks].sum(axis=1)
        signs = np.array([[1 if (mask >> i) & 1 else -1 for i in range(len(order_docs))] for mask in range(2 ** len(order_docs))])
        flipped = signs @ sums
        return {"n": int(len(diff)), "delta": float(diff.mean()), "ci95": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
                "p05_one_sided": float(np.percentile(boot, 5)), "sign_flip_p": float(np.mean(flipped >= sums.sum() - 1e-12))}

    def fixed_broken(t, c):
        return int(((t == 1) & (c == 0)).sum()), int(((t == 0) & (c == 1)).sum())

    b1s, aga, agb = per_question("B1s"), per_question("AGa"), per_question("AGb")
    ag = (aga + agb) / 2
    primary = contrast(ag, b1s)
    per_rep = {rep: {"delta": float((x - b1s).mean()), "fixed": fixed_broken(x, b1s)[0], "broken": fixed_broken(x, b1s)[1]}
               for rep, x in (("AGa", aga), ("AGb", agb))}
    rec = {arm: per_question(arm, "recall_seen", "labelled") for arm in ARMS}
    mechanism = contrast((rec["AGa"] + rec["AGb"]) / 2, rec["B1s"])
    inc = set(scores.loc[scores["incomplete_at_4"], "qid"])
    where = {name: contrast(ag[ag.index.isin(sel) if keep else ~ag.index.isin(sel)], b1s[b1s.index.isin(sel) if keep else ~b1s.index.isin(sel)])
             for name, sel, keep in (("evidence incomplete in top 4 (39)", inc, True), ("evidence complete in top 4 or unlabelled", inc, False))}
    buckets = scores[scores["answerable"] & scores["labelled"]].assign(
        bucket=lambda f: pd.cut(f["n_evidence"], [0, 1, 3, 100], labels=["1 page", "2-3 pages", "4+ pages"]))
    bucket_table = buckets.pivot_table(index="bucket", columns="arm", values="s1_correct", aggfunc=["sum", "count"], observed=True)
    flips = int((aga != agb).sum())
    e8_lb1 = e8[(e8["arm"] == "L-B1") & e8["answerable"]].set_index("qid")["s1_correct"].astype(float).sort_index()
    e9_b1s = e9_table[(e9_table["arm"] == "B1s") & e9_table["answerable"]].set_index("qid")["s1_correct"].astype(float).sort_index()
    drift = {"B1s vs frozen E8 L-B1": int((b1s != e8_lb1).sum()), "B1s vs frozen E9 B1s": int((b1s != e9_b1s).sum()),
             "frozen E8 L-B1 correct": int(e8_lb1.sum()), "frozen E9 B1s correct": int(e9_b1s.sum())}

    # The frozen decision rules, in order (spec decision_rules_in_order).
    base = arm_table.loc["B1s"]
    guardrails = {arm: {"abstentions": arm_table.at[arm, "correct abstentions / 16"] >= base["correct abstentions / 16"] - 2,
                        "citations": arm_table.at[arm, "citation validity"] >= 0.99,
                        "schema": arm_table.at[arm, "schema validity"] >= 0.99} for arm in ("AGa", "AGb")}
    guardrails_pass = all(all(g.values()) for g in guardrails.values())
    delta = primary["delta"]
    engineering = delta >= 0.05 and all(r["delta"] >= 0.05 for r in per_rep.values())
    broken_rule = any(r["broken"] > r["fixed"] / 2 for r in per_rep.values())
    mechanism_ok = mechanism["delta"] > 0 and mechanism["p05_one_sided"] > 0
    ag_cost = float(scores.loc[scores["arm"].isin(["AGa", "AGb"]), "cost_usd"].mean())
    cost_ok = ag_cost <= 0.01
    statistics_ok = primary["p05_one_sided"] > 0 and primary["sign_flip_p"] <= 0.05
    if not guardrails_pass or delta <= -0.05:
        verdict = "regression"
    elif not engineering:
        verdict = "no_meaningful_gain"
    elif broken_rule:
        verdict = "regression"
    elif not mechanism_ok:
        verdict = "gain_without_mechanism"
    elif not cost_ok:
        verdict = "too_expensive"
    elif statistics_ok and flips <= 10:
        verdict = "go"
    else:
        verdict = "promising_but_uncertain"

    decision = {"primary contrast": "AG (AGa, AGb averaged) - B1s, S-v1, 102 answerable", "delta": round(delta, 4),
                "ci95": [round(x, 4) for x in primary["ci95"]], "bootstrap 5th percentile": round(primary["p05_one_sided"], 4),
                "sign-flip p": round(primary["sign_flip_p"], 5), "per replicate": per_rep, "guardrails pass": guardrails_pass,
                "engineering criterion (+0.05 average and each replicate)": engineering, "broken > fixed/2": broken_rule,
                "mechanism (evidence recall of pages seen)": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in mechanism.items()},
                "agent cost per question (USD)": round(ag_cost, 6), "cost <= $0.01": cost_ok, "statistics hold": statistics_ok,
                "AGa vs AGb flips": flips, "VERDICT (S-v1, frozen rules)": verdict}
    display(pd.Series({k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in decision.items()}, name="E10 decision").to_frame())
    display(pd.DataFrame(where).T.round(4))
    display(bucket_table)
    display(pd.Series(drift, name="session drift (descriptive)").to_frame())

    # Trajectories (agent arms): how the agent spent its budget.
    agent_rows = scores[scores["arm"] != "B1s"]
    trajectory = {"finished at call 1": float(agent_rows["finished_at_call_1"].mean()), "calls (mean)": float(agent_rows["calls"].mean()),
                  "calls distribution": agent_rows["calls"].value_counts().sort_index().to_dict(),
                  "searches (mean)": float(agent_rows["searches"].mean()), "reads (mean)": float(agent_rows["reads"].mean()),
                  "repeated searches": int(agent_rows["repeats"].sum()), "protocol / argument errors": int(agent_rows["protocol_errors"].sum()),
                  "forced finishes": int(agent_rows["forced_finish"].sum()), "page budget exhausted": int((agent_rows["pages_seen"] >= MAX_PAGES).sum()),
                  "billed minus estimated prompt tokens (agent calls): median": float(estimate_gap.median()),
                  "... max": float(estimate_gap.max()), "B1s prompt totals exact": b1_exact}
    display(pd.Series({k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in trajectory.items()}, name="agent trajectories").to_frame())

    # S-v2 (secondary): bounds while items are pending; a point estimate once the owner has labelled them.
    def v2_correct(arm, pending_as):
        frame = scores[(scores["arm"] == arm) & scores["answerable"]].set_index("qid").sort_index()
        final = frame["v2_final"].replace({"pending": pending_as})
        return (final[final != "unsure"] == "correct").astype(float)

    v2 = {}
    for bound in ("wrong", "correct"):
        b, a = v2_correct("B1s", bound), (v2_correct("AGa", bound) + v2_correct("AGb", bound)) / 2
        common = b.index.intersection(a.index)
        v2[f"pending as {bound}"] = {"B1s": float(b[common].sum()), "AG (avg)": float(a[common].sum()),
                                     "AG - B1s": float((a[common] - b[common]).mean()), "n": len(common)}
    display(pd.DataFrame(v2).T.round(4))

    scores.drop(columns=["prediction", "identity"]).to_csv(OUT_DIR / "evaluation_per_question.csv", index=False)
    E10_SUMMARY = {"run": EVAL_RUN.name, "arms": arm_table.round(6).to_dict(orient="index"), "decision": decision,
                   "where": where, "bucket_table": json.loads(bucket_table.to_json()), "drift": drift, "trajectory": trajectory,
                   "s_v2": v2, "s_v2_pending_items": int(len(queue)),
                   "spend_usd": {"observed_list_price": round(sum(observed_cost(c) for c in calls), 9),
                                 "smoke_observed_list_price": round(sum(observed_cost(c) for p in RUNS_DIR.glob("smoke_*/calls.jsonl") for c in read_jsonl(p)), 9)}}
    (OUT_DIR / "evaluation_summary.json").write_text(json.dumps(E10_SUMMARY, indent=2, default=str) + "\n")
    print(f"Evaluated {EVAL_RUN.relative_to(PROJECT_ROOT)}: verdict {verdict}. Tables saved to {OUT_DIR.relative_to(PROJECT_ROOT)}.")

,S-v1 correct / 102,correct abstentions / 16,citation validity,schema validity,S-v2 correct (auto + owner),S-v2 pending,evidence recall of pages seen (101),evidence complete in pages seen (101),pages seen (mean),calls (mean),searches (mean),prompt tokens (mean),cached share,cost USD per question,latency s p50,latency s p90
B1s,45.0,8.0,1.0,1.0,46.0,3.0,0.7259,62.0,4.0000,1.0000,0.0000,3977.1525,0.0000,0.0004,2.2745,3.8592
AGa,53.0,9.0,1.0,1.0,50.0,8.0,0.8676,77.0,6.7542,3.3475,1.2034,21569.5254,0.2378,0.0018,5.4220,17.3022
AGb,53.0,12.0,1.0,1.0,51.0,10.0,0.8635,78.0,6.8136,3.2712,1.1356,20929.5593,0.2472,0.0017,5.7280,14.9205


,E10 decision
primary contrast,"AG (AGa, AGb averaged) - B1s, S-v1, 102 answerable"
delta,0.0784
ci95,"[0.0, 0.1812]"
bootstrap 5th percentile,0.0098
sign-flip p,0.05273
per replicate,"{""AGa"": {""delta"": 0.0784313725490196, ""fixed"": 13, ""broken"": 5}, ""AGb"": {""delta"": 0.07..."
guardrails pass,True
engineering criterion (+0.05 average and each replicate),True
broken > fixed/2,True
mechanism (evidence recall of pages seen),"{""n"": 101, ""delta"": 0.1396, ""ci95"": [0.08370968980343982, 0.19846491228070176], ""p05_o..."


,n,delta,ci95,p05_one_sided,sign_flip_p
evidence incomplete in top 4 (39),39,0.25641,"[0.136364, 0.5]",0.150943,0.000977
evidence complete in top 4 or unlabelled,63,-0.031746,"[-0.129318, 0.084746]",-0.114754,0.75


sum         count        
arm       AGa AGb B1s   AGa AGb B1s
bucket                             
1 page     36  36  33    55  55  55
2-3 pages  16  16  10    40  40  40
4+ pages    1   1   1     6   6   6

,session drift (descriptive)
B1s vs frozen E8 L-B1,5
B1s vs frozen E9 B1s,5
frozen E8 L-B1 correct,46
frozen E9 B1s correct,44


,agent trajectories
finished at call 1,0.292373
calls (mean),3.309322
calls distribution,"{""1"": 69, ""2"": 37, ""3"": 29, ""4"": 3, ""5"": 49, ""6"": 49}"
searches (mean),1.169492
reads (mean),1.135593
repeated searches,1
protocol / argument errors,0
forced finishes,49
page budget exhausted,19
billed minus estimated prompt tokens (agent calls): median,262.0


,B1s,AG (avg),AG - B1s,n
pending as wrong,46.0,50.5,0.0446,101.0
pending as correct,49.0,59.5,0.1040,101.0


Evaluated experiments/frozen/e10/full_20261127T103158Z_3049dc6c: verdict regression. Tables saved to data/processed/mmlongbench_doc/e10.


## Results and interpretation (E10, development data)

Frozen package `experiments/frozen/e10/`. Independent audit `AUDIT.json`: 31/31 checks.

**Run.** 354 items, 899 calls, all first-attempt successes on one served model (`gpt-6-luna`). Spend:
- full run: $0.455862;
- smoke: $0.017513, including the $0.0011 charged for a locally failed first attempt;
- total: **$0.473** of the $3.50 ceiling.

| S-v1, 118 questions | B1s | AGa | AGb |
|---|---:|---:|---:|
| correct / 102 answerable | 45 | **53** | **53** |
| correct abstentions / 16 | 8 | 9 | 12 |
| citation validity, schema validity | 1.000, 1.000 | 1.000, 1.000 | 1.000, 1.000 |
| evidence recall of pages seen (101) | 0.726 | **0.868** | **0.864** |
| evidence complete in pages seen (101) | 62 | 77 | 78 |
| calls / pages seen per question | 1 / 4 | 3.35 / 6.75 | 3.27 / 6.81 |
| cost per question, latency p50 / p90 | $0.0004, 2.3 / 3.9 s | $0.0018, 5.4 / 17.3 s | $0.0017, 5.7 / 14.9 s |

**Primary contrast:**
- AG − B1s = **+0.078** (+8 questions), 95 % interval [0.000, +0.181], bootstrap 5th percentile +0.010, exact sign-flip p = 0.053.
- Fixed / broken: AGa 13 / 5, AGb 17 / 9.
- AGa and AGb disagree on 12 questions.

**Frozen verdict: `regression`.** The pre-registered rule "broken > fixed / 2 in either replicate" fires on AGb (9 > 8.5). Without that rule the verdict would still not be `go`: p = 0.053 > 0.05 and 12 flips > 10. So the outcome would be `promising_but_uncertain`. The verdict is reported as frozen, and the name of the rule should not be read as "the agent is worse". The agent is **+8 questions net in both replicates**.

**Mechanism and location (H2, H3): both strongly supported.**
- **H2.** Evidence recall of the pages seen rose by +0.140 [+0.084, +0.198], sign-flip p = 0.0002.
- **H3, where the evidence was incomplete in B1's top 4** (39 questions): +0.256 [+0.136, +0.500], p = 0.001. 9 of AGa's 13 fixes and 11 of AGb's 17 are there.
- **H3, everywhere else:** −0.032 [−0.129, +0.085]. **Every broken question had its evidence complete in B1's top 4.**

**Why it breaks questions (all 5 questions broken in both replicates, read by hand):**
1. **Over-exploration** (mmlb-0048, mmlb-0599). The answer was already in view, but the agent searched and read 7–8 more pages, then miscounted or added colours from other pages.
2. **A different reading of the same 4 pages** (mmlb-0201, mmlb-1014). No new page was read, yet the answer changed. The agent's prompt and tool-call mode alter the reading itself, a confound the warm start cannot remove.
3. **Answer format** (mmlb-1025: "1,358,000 shares" instead of 1358000). S-v1 scores the text form wrong. This is a scoring artefact more than an error.

**Why it fixes questions** (5 sampled from the 11 fixed in both replicates). The agent wrote targeted queries and found pages ColQwen2 had ranked low for the original question. For example:
- "income tax provision … effective tax rate reconciliation" found pages 64 and 38 (mmlb-1055);
- "student cannot be found … whom to contact" found page 25 (mmlb-0990).

**Exploratory, not pre-registered:**
- **The agent's own stopping decision is a usable router.** 29 % of agent episodes finished at call 1 and were 76 % correct, against 41 % for the episodes that searched.
- **E9's U3 trigger is a worse router.** Using the agent only where U3 fires gives 46 (AGa) and 49 (AGb) of 102, below always running the agent (53).

**Session drift.** B1s scored 45, against 46 for frozen E8 L-B1 and 44 for frozen E9 B1s: 5 flips each, on byte-identical requests.

**S-v2 (secondary).** 17 new blind gray items await the owner's labels (`adjudication/blind_queue_unlabelled.csv`). Bounds while they are pending: AG − B1s from +0.045 (all counted wrong) to +0.104 (all counted correct).

**What this means.** Searching works: it finds the missing evidence and turns it into answers, concentrated where B1 fails. The cost is a smaller loss on questions B1 already answers, from over-exploration and from the change of reading mode. The next experiment should keep the search and remove that loss. Options:
- stop when the warm-start pages suffice: a stricter finish-first instruction, or the B1 answer as a fallback;
- read in B1's mode for the final answer.

It should be tested on a new benchmark, since these 118 questions are development data.